In [6]:
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.5


In [7]:
data = [
    {
        "question": "What is Python?",
        "expected_answer": "Python is a high-level programming language known for its simple syntax and readability.",
        "candidate_answer": "Python is a high-level programming language with simple and readable syntax.",
        "question_type": "Python",
        "quality_label": "Excellent"
    },
    {
        "question": "What is Python?",
        "expected_answer": "Python is a high-level programming language known for its simple syntax and readability.",
        "candidate_answer": "Python is a programming language that is easy to learn.",
        "question_type": "Python",
        "quality_label": "Good"
    },
    {
        "question": "What is Python?",
        "expected_answer": "Python is a high-level programming language known for its simple syntax and readability.",
        "candidate_answer": "Python is used for programming.",
        "question_type": "Python",
        "quality_label": "Average"
    },
    {
        "question": "What is Python?",
        "expected_answer": "Python is a high-level programming language known for its simple syntax and readability.",
        "candidate_answer": "I have heard about Python.",
        "question_type": "Python",
        "quality_label": "Poor"
    }
]

df = pd.DataFrame(data)

df

,question,expected_answer,candidate_answer,question_type,quality_label
0,What is Python?,Python is a high-level programming language kn...,Python is a high-level programming language wi...,Python,Excellent
1,What is Python?,Python is a high-level programming language kn...,Python is a programming language that is easy ...,Python,Good
2,What is Python?,Python is a high-level programming language kn...,Python is used for programming.,Python,Average
3,What is Python?,Python is a high-level programming language kn...,I have heard about Python.,Python,Poor


In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4859.11it/s]


Sentence Transformer loaded successfully!


In [10]:
expected = df.loc[0, "expected_answer"]
candidate = df.loc[0, "candidate_answer"]

print("Expected Answer:")
print(expected)

print("\nCandidate Answer:")
print(candidate)

Expected Answer:
Python is a high-level programming language known for its simple syntax and readability.

Candidate Answer:
Python is a high-level programming language with simple and readable syntax.


In [11]:
expected_embedding = model.encode(expected)
candidate_embedding = model.encode(candidate)

print("Expected embedding shape:", expected_embedding.shape)
print("Candidate embedding shape:", candidate_embedding.shape)

Expected embedding shape: (384,)
Candidate embedding shape: (384,)


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
similarity = cosine_similarity(
    [expected_embedding],
    [candidate_embedding]
)

semantic_score = similarity[0][0]

print("Semantic Similarity Score:", semantic_score)

Semantic Similarity Score: 0.98095953


In [14]:
semantic_scores = []

In [15]:
for index, row in df.iterrows():

    expected = row["expected_answer"]
    candidate = row["candidate_answer"]

    expected_embedding = model.encode(expected)
    candidate_embedding = model.encode(candidate)

    similarity = cosine_similarity(
        [expected_embedding],
        [candidate_embedding]
    )

    semantic_score = similarity[0][0]

    semantic_scores.append(semantic_score)

In [16]:
df["semantic_score"] = semantic_scores

df

,question,expected_answer,candidate_answer,question_type,quality_label,semantic_score
0,What is Python?,Python is a high-level programming language kn...,Python is a high-level programming language wi...,Python,Excellent,0.980960
1,What is Python?,Python is a high-level programming language kn...,Python is a programming language that is easy ...,Python,Good,0.861897
2,What is Python?,Python is a high-level programming language kn...,Python is used for programming.,Python,Average,0.804986
3,What is Python?,Python is a high-level programming language kn...,I have heard about Python.,Python,Poor,0.684221


In [17]:
python_keywords = [
    "python",
    "high-level",
    "programming language",
    "syntax",
    "readability"
]

df["keywords"] = [python_keywords] * len(df)

df[["candidate_answer", "keywords", "quality_label"]]

,candidate_answer,keywords,quality_label
0,Python is a high-level programming language wi...,"[python, high-level, programming language, syn...",Excellent
1,Python is a programming language that is easy ...,"[python, high-level, programming language, syn...",Good
2,Python is used for programming.,"[python, high-level, programming language, syn...",Average
3,I have heard about Python.,"[python, high-level, programming language, syn...",Poor


In [18]:
def calculate_keyword_score(answer, keywords):

    answer = answer.lower()

    matched_keywords = 0

    for keyword in keywords:

        if keyword.lower() in answer:
            matched_keywords += 1

    score = matched_keywords / len(keywords)

    return score

In [19]:
df["keyword_score"] = df.apply(
    lambda row: calculate_keyword_score(
        row["candidate_answer"],
        row["keywords"]
    ),
    axis=1
)

df[["candidate_answer", "keyword_score", "quality_label"]]

,candidate_answer,keyword_score,quality_label
0,Python is a high-level programming language wi...,0.8,Excellent
1,Python is a programming language that is easy ...,0.4,Good
2,Python is used for programming.,0.2,Average
3,I have heard about Python.,0.2,Poor


In [20]:
df["answer_length"] = df["candidate_answer"].apply(
    lambda answer: len(answer.split())
)

df[["candidate_answer", "answer_length", "quality_label"]]

,candidate_answer,answer_length,quality_label
0,Python is a high-level programming language wi...,11,Excellent
1,Python is a programming language that is easy ...,10,Good
2,Python is used for programming.,5,Average
3,I have heard about Python.,5,Poor


In [21]:
question_bank = [

    {
        "question": "What is Python?",
        "expected_answer": "Python is a high-level programming language known for its simple syntax and readability.",
        "keywords": [
            "python",
            "high-level",
            "programming language",
            "syntax",
            "readability"
        ],
        "question_type": "Python"
    },

    {
        "question": "What is machine learning?",
        "expected_answer": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions.",
        "keywords": [
            "machine learning",
            "artificial intelligence",
            "learn",
            "patterns",
            "data",
            "predictions"
        ],
        "question_type": "Machine Learning"
    },

    {
        "question": "What is a primary key in SQL?",
        "expected_answer": "A primary key is a column or combination of columns that uniquely identifies each record in a database table.",
        "keywords": [
            "primary key",
            "unique",
            "record",
            "table",
            "database"
        ],
        "question_type": "SQL"
    },

    {
        "question": "What is object-oriented programming?",
        "expected_answer": "Object-oriented programming is a programming paradigm based on objects and classes that combines data and behavior.",
        "keywords": [
            "object-oriented programming",
            "objects",
            "classes",
            "data",
            "behavior",
            "paradigm"
        ],
        "question_type": "OOP"
    },

    {
        "question": "What is data preprocessing?",
        "expected_answer": "Data preprocessing is the process of cleaning and transforming raw data into a suitable format for analysis or machine learning.",
        "keywords": [
            "data preprocessing",
            "cleaning",
            "transforming",
            "raw data",
            "machine learning"
        ],
        "question_type": "Data Science"
    }
]

In [22]:
print("Number of questions:", len(question_bank))

for question in question_bank:
    print("\nQuestion:", question["question"])
    print("Type:", question["question_type"])
    print("Keywords:", question["keywords"])

Number of questions: 5

Question: What is Python?
Type: Python
Keywords: ['python', 'high-level', 'programming language', 'syntax', 'readability']

Question: What is machine learning?
Type: Machine Learning
Keywords: ['machine learning', 'artificial intelligence', 'learn', 'patterns', 'data', 'predictions']

Question: What is a primary key in SQL?
Type: SQL
Keywords: ['primary key', 'unique', 'record', 'table', 'database']

Question: What is object-oriented programming?
Type: OOP
Keywords: ['object-oriented programming', 'objects', 'classes', 'data', 'behavior', 'paradigm']

Question: What is data preprocessing?
Type: Data Science
Keywords: ['data preprocessing', 'cleaning', 'transforming', 'raw data', 'machine learning']


In [23]:
candidate_answers = [

    # Python
    {
        "question": "What is Python?",
        "candidate_answer": "Python is a high-level programming language known for its simple syntax and readability. It is widely used for web development, data science, automation, and machine learning.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is Python?",
        "candidate_answer": "Python is a high-level programming language with simple syntax. It is easy to learn and is used in many applications.",
        "quality_label": "Good"
    },
    {
        "question": "What is Python?",
        "candidate_answer": "Python is a programming language used to develop applications.",
        "quality_label": "Average"
    },
    {
        "question": "What is Python?",
        "candidate_answer": "I have heard about Python.",
        "quality_label": "Poor"
    },

    # Machine Learning
    {
        "question": "What is machine learning?",
        "candidate_answer": "Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is machine learning?",
        "candidate_answer": "Machine learning is a part of artificial intelligence where computers learn from data and make predictions.",
        "quality_label": "Good"
    },
    {
        "question": "What is machine learning?",
        "candidate_answer": "Machine learning allows computers to learn from data.",
        "quality_label": "Average"
    },
    {
        "question": "What is machine learning?",
        "candidate_answer": "Machine learning is related to computers.",
        "quality_label": "Poor"
    },

    # SQL
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key is a column or combination of columns that uniquely identifies each record in a database table. It cannot contain duplicate values.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key uniquely identifies each record in a database table.",
        "quality_label": "Good"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key identifies data in a table.",
        "quality_label": "Average"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "It is something used in SQL.",
        "quality_label": "Poor"
    },

    # OOP
    {
        "question": "What is object-oriented programming?",
        "candidate_answer": "Object-oriented programming is a programming paradigm based on objects and classes. It combines data and behavior and commonly uses concepts such as inheritance, encapsulation, polymorphism, and abstraction.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is object-oriented programming?",
        "candidate_answer": "Object-oriented programming is a programming approach that uses objects and classes to organize data and behavior.",
        "quality_label": "Good"
    },
    {
        "question": "What is object-oriented programming?",
        "candidate_answer": "OOP is a programming method that uses objects.",
        "quality_label": "Average"
    },
    {
        "question": "What is object-oriented programming?",
        "candidate_answer": "It is related to programming.",
        "quality_label": "Poor"
    },

    # Data Science
    {
        "question": "What is data preprocessing?",
        "candidate_answer": "Data preprocessing is the process of cleaning and transforming raw data into a suitable format for analysis or machine learning. It may include handling missing values, removing duplicates, encoding categorical data, and scaling numerical features.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is data preprocessing?",
        "candidate_answer": "Data preprocessing involves cleaning and transforming raw data before using it for analysis or machine learning.",
        "quality_label": "Good"
    },
    {
        "question": "What is data preprocessing?",
        "candidate_answer": "Data preprocessing means cleaning data before analysis.",
        "quality_label": "Average"
    },
    {
        "question": "What is data preprocessing?",
        "candidate_answer": "It is something done with data.",
        "quality_label": "Poor"
    }
]

print("Number of candidate answers:", len(candidate_answers))

Number of candidate answers: 20


In [24]:
question_info = {
    item["question"]: item
    for item in question_bank
}

print(question_info.keys())

dict_keys(['What is Python?', 'What is machine learning?', 'What is a primary key in SQL?', 'What is object-oriented programming?', 'What is data preprocessing?'])


In [25]:
dataset = []

for answer in candidate_answers:

    question = answer["question"]

    info = question_info[question]

    dataset.append({
        "question": question,
        "expected_answer": info["expected_answer"],
        "candidate_answer": answer["candidate_answer"],
        "keywords": info["keywords"],
        "question_type": info["question_type"],
        "quality_label": answer["quality_label"]
    })

print("Total dataset samples:", len(dataset))

Total dataset samples: 20


In [26]:
df = pd.DataFrame(dataset)

df

,question,expected_answer,candidate_answer,keywords,question_type,quality_label
0,What is Python?,Python is a high-level programming language kn...,Python is a high-level programming language kn...,"[python, high-level, programming language, syn...",Python,Excellent
1,What is Python?,Python is a high-level programming language kn...,Python is a high-level programming language wi...,"[python, high-level, programming language, syn...",Python,Good
2,What is Python?,Python is a high-level programming language kn...,Python is a programming language used to devel...,"[python, high-level, programming language, syn...",Python,Average
3,What is Python?,Python is a high-level programming language kn...,I have heard about Python.,"[python, high-level, programming language, syn...",Python,Poor
4,What is machine learning?,Machine learning is a branch of artificial int...,Machine learning is a branch of artificial int...,"[machine learning, artificial intelligence, le...",Machine Learning,Excellent
5,What is machine learning?,Machine learning is a branch of artificial int...,Machine learning is a part of artificial intel...,"[machine learning, artificial intelligence, le...",Machine Learning,Good
6,What is machine learning?,Machine learning is a branch of artificial int...,Machine learning allows computers to learn fro...,"[machine learning, artificial intelligence, le...",Machine Learning,Average
7,What is machine learning?,Machine learning is a branch of artificial int...,Machine learning is related to computers.,"[machine learning, artificial intelligence, le...",Machine Learning,Poor
8,What is a primary key in SQL?,A primary key is a column or combination of co...,A primary key is a column or combination of co...,"[primary key, unique, record, table, database]",SQL,Excellent
9,What is a primary key in SQL?,A primary key is a column or combination of co...,A primary key uniquely identifies each record ...,"[primary key, unique, record, table, database]",SQL,Good


In [27]:
semantic_scores = []

for index, row in df.iterrows():

    expected_embedding = model.encode(row["expected_answer"])
    candidate_embedding = model.encode(row["candidate_answer"])

    similarity = cosine_similarity(
        [expected_embedding],
        [candidate_embedding]
    )

    semantic_scores.append(similarity[0][0])

df["semantic_score"] = semantic_scores

In [28]:
df["keyword_score"] = df.apply(
    lambda row: calculate_keyword_score(
        row["candidate_answer"],
        row["keywords"]
    ),
    axis=1
)

In [29]:
df["answer_length"] = df["candidate_answer"].apply(
    lambda answer: len(answer.split())
)

In [30]:
df[
    [
        "question",
        "candidate_answer",
        "semantic_score",
        "keyword_score",
        "answer_length",
        "quality_label"
    ]
]

,question,candidate_answer,semantic_score,keyword_score,answer_length,quality_label
0,What is Python?,Python is a high-level programming language kn...,0.941670,1.000000,26,Excellent
1,What is Python?,Python is a high-level programming language wi...,0.928936,0.800000,20,Good
2,What is Python?,Python is a programming language used to devel...,0.812251,0.400000,9,Average
3,What is Python?,I have heard about Python.,0.684221,0.200000,5,Poor
4,What is machine learning?,Machine learning is a branch of artificial int...,0.977802,1.000000,28,Excellent
5,What is machine learning?,Machine learning is a part of artificial intel...,0.934238,0.833333,16,Good
6,What is machine learning?,Machine learning allows computers to learn fro...,0.840534,0.500000,8,Average
7,What is machine learning?,Machine learning is related to computers.,0.825154,0.333333,6,Poor
8,What is a primary key in SQL?,A primary key is a column or combination of co...,0.929647,1.000000,24,Excellent
9,What is a primary key in SQL?,A primary key uniquely identifies each record ...,0.887716,1.000000,11,Good


In [31]:
additional_questions = [

    {
        "question": "What is a list in Python?",
        "expected_answer": "A list is a mutable ordered collection in Python that can store multiple values and allows duplicate elements.",
        "keywords": [
            "list",
            "mutable",
            "ordered",
            "collection",
            "duplicate"
        ],
        "question_type": "Python"
    },

    {
        "question": "What is supervised learning?",
        "expected_answer": "Supervised learning is a machine learning approach where a model learns from labeled training data to make predictions on new data.",
        "keywords": [
            "supervised learning",
            "machine learning",
            "labeled",
            "training data",
            "predictions"
        ],
        "question_type": "Machine Learning"
    },

    {
        "question": "What is unsupervised learning?",
        "expected_answer": "Unsupervised learning is a machine learning approach that finds patterns or structures in data without using labeled target values.",
        "keywords": [
            "unsupervised learning",
            "machine learning",
            "patterns",
            "structures",
            "unlabeled"
        ],
        "question_type": "Machine Learning"
    },

    {
        "question": "What is overfitting in machine learning?",
        "expected_answer": "Overfitting occurs when a machine learning model learns the training data too closely, including noise, and performs poorly on unseen data.",
        "keywords": [
            "overfitting",
            "model",
            "training data",
            "noise",
            "unseen data"
        ],
        "question_type": "Machine Learning"
    },

    {
        "question": "What is SQL?",
        "expected_answer": "SQL is a structured query language used to store, retrieve, manipulate, and manage data in relational databases.",
        "keywords": [
            "SQL",
            "structured query language",
            "relational database",
            "retrieve",
            "manage data"
        ],
        "question_type": "SQL"
    },

    {
        "question": "What is database normalization?",
        "expected_answer": "Database normalization is the process of organizing data in tables to reduce redundancy and improve data integrity.",
        "keywords": [
            "normalization",
            "database",
            "tables",
            "redundancy",
            "data integrity"
        ],
        "question_type": "SQL"
    },

    {
        "question": "What is inheritance in OOP?",
        "expected_answer": "Inheritance is an object-oriented programming concept where a class can acquire properties and methods from another class.",
        "keywords": [
            "inheritance",
            "object-oriented programming",
            "class",
            "properties",
            "methods"
        ],
        "question_type": "OOP"
    },

    {
        "question": "What is encapsulation in OOP?",
        "expected_answer": "Encapsulation is an object-oriented programming concept that combines data and methods inside a class and restricts direct access to internal data.",
        "keywords": [
            "encapsulation",
            "object-oriented programming",
            "data",
            "methods",
            "class"
        ],
        "question_type": "OOP"
    },

    {
        "question": "What is exploratory data analysis?",
        "expected_answer": "Exploratory data analysis is the process of examining and visualizing a dataset to understand patterns, relationships, distributions, and anomalies.",
        "keywords": [
            "exploratory data analysis",
            "dataset",
            "visualizing",
            "patterns",
            "relationships",
            "anomalies"
        ],
        "question_type": "Data Science"
    },

    {
        "question": "What is feature engineering?",
        "expected_answer": "Feature engineering is the process of creating, transforming, or selecting useful input features from raw data to improve machine learning performance.",
        "keywords": [
            "feature engineering",
            "features",
            "raw data",
            "transforming",
            "machine learning"
        ],
        "question_type": "Data Science"
    }
]

In [32]:
question_bank.extend(additional_questions)

print("Total questions:", len(question_bank))

Total questions: 15


In [33]:
candidate_answers = [

    # ---------------- PYTHON ----------------

    {
        "question": "What is Python?",
        "candidate_answer": "Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is Python?",
        "candidate_answer": "Python is a high-level programming language with simple syntax. It is easy to learn and is used in many different applications.",
        "quality_label": "Good"
    },
    {
        "question": "What is Python?",
        "candidate_answer": "Python is a programming language used to develop software and applications.",
        "quality_label": "Average"
    },
    {
        "question": "What is Python?",
        "candidate_answer": "Python is a language used for programming.",
        "quality_label": "Poor"
    },

    {
        "question": "What is a list in Python?",
        "candidate_answer": "A list is a mutable and ordered collection in Python that can store multiple values, including duplicate elements and values of different data types.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a list in Python?",
        "candidate_answer": "A Python list is an ordered collection that can store multiple values and can be modified after creation.",
        "quality_label": "Good"
    },
    {
        "question": "What is a list in Python?",
        "candidate_answer": "A list stores multiple values in Python.",
        "quality_label": "Average"
    },
    {
        "question": "What is a list in Python?",
        "candidate_answer": "A list is something used in Python.",
        "quality_label": "Poor"
    },

    # ---------------- MACHINE LEARNING ----------------

    {
        "question": "What is machine learning?",
        "candidate_answer": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is machine learning?",
        "candidate_answer": "Machine learning is a part of artificial intelligence where computers learn from data and use the learned patterns to make predictions.",
        "quality_label": "Good"
    },
    {
        "question": "What is machine learning?",
        "candidate_answer": "Machine learning allows computers to learn from data.",
        "quality_label": "Average"
    },
    {
        "question": "What is machine learning?",
        "candidate_answer": "Machine learning is related to computers.",
        "quality_label": "Poor"
    },

    {
        "question": "What is supervised learning?",
        "candidate_answer": "Supervised learning is a machine learning approach where a model learns from labeled training data and uses those learned patterns to make predictions on new unseen data.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is supervised learning?",
        "candidate_answer": "Supervised learning trains a machine learning model using labeled data so that it can predict outcomes for new data.",
        "quality_label": "Good"
    },
    {
        "question": "What is supervised learning?",
        "candidate_answer": "Supervised learning uses data to train a model and make predictions.",
        "quality_label": "Average"
    },
    {
        "question": "What is supervised learning?",
        "candidate_answer": "It is a type of machine learning.",
        "quality_label": "Poor"
    },

    {
        "question": "What is unsupervised learning?",
        "candidate_answer": "Unsupervised learning is a machine learning approach that discovers patterns, groups, or structures in data without using labeled target values.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is unsupervised learning?",
        "candidate_answer": "Unsupervised learning finds useful patterns or groups in data when labeled output values are not available.",
        "quality_label": "Good"
    },
    {
        "question": "What is unsupervised learning?",
        "candidate_answer": "It allows a model to find patterns in data without labels.",
        "quality_label": "Average"
    },
    {
        "question": "What is unsupervised learning?",
        "candidate_answer": "It is another type of machine learning.",
        "quality_label": "Poor"
    },

    {
        "question": "What is overfitting in machine learning?",
        "candidate_answer": "Overfitting occurs when a machine learning model learns the training data too closely, including noise, and therefore performs very well on training data but poorly on unseen data.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is overfitting in machine learning?",
        "candidate_answer": "Overfitting happens when a model fits the training data too closely and performs poorly when given new unseen data.",
        "quality_label": "Good"
    },
    {
        "question": "What is overfitting in machine learning?",
        "candidate_answer": "Overfitting means a model learns the training data too much.",
        "quality_label": "Average"
    },
    {
        "question": "What is overfitting in machine learning?",
        "candidate_answer": "It is a problem with machine learning models.",
        "quality_label": "Poor"
    },

    # ---------------- SQL ----------------

    {
        "question": "What is SQL?",
        "candidate_answer": "SQL stands for Structured Query Language and is used to store, retrieve, manipulate, and manage data in relational database systems.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is SQL?",
        "candidate_answer": "SQL is a language used to communicate with relational databases and perform operations on stored data.",
        "quality_label": "Good"
    },
    {
        "question": "What is SQL?",
        "candidate_answer": "SQL is used to work with databases.",
        "quality_label": "Average"
    },
    {
        "question": "What is SQL?",
        "candidate_answer": "SQL is a programming language.",
        "quality_label": "Poor"
    },

    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key is a column or combination of columns that uniquely identifies each record in a database table. It cannot contain duplicate values and normally cannot contain NULL values.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key uniquely identifies each record in a database table and prevents duplicate values.",
        "quality_label": "Good"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key identifies records in a table.",
        "quality_label": "Average"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "It is something used in SQL.",
        "quality_label": "Poor"
    },

    {
        "question": "What is database normalization?",
        "candidate_answer": "Database normalization is the process of organizing data into related tables to reduce redundancy, avoid data anomalies, and improve data integrity.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is database normalization?",
        "candidate_answer": "Normalization organizes database data into tables to reduce duplicate data and improve consistency.",
        "quality_label": "Good"
    },
    {
        "question": "What is database normalization?",
        "candidate_answer": "Normalization is a way of organizing data in a database.",
        "quality_label": "Average"
    },
    {
        "question": "What is database normalization?",
        "candidate_answer": "It is related to databases.",
        "quality_label": "Poor"
    },

    # ---------------- OOP ----------------

    {
        "question": "What is object-oriented programming?",
        "candidate_answer": "Object-oriented programming is a programming paradigm based on objects and classes. It combines data and behavior and commonly uses concepts such as inheritance, encapsulation, polymorphism, and abstraction.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is object-oriented programming?",
        "candidate_answer": "OOP is a programming approach that uses objects and classes to organize data and behavior.",
        "quality_label": "Good"
    },
    {
        "question": "What is object-oriented programming?",
        "candidate_answer": "OOP is a programming method that uses objects.",
        "quality_label": "Average"
    },
    {
        "question": "What is object-oriented programming?",
        "candidate_answer": "It is related to programming.",
        "quality_label": "Poor"
    },

    {
        "question": "What is inheritance in OOP?",
        "candidate_answer": "Inheritance is an object-oriented programming concept where a class can acquire properties and methods from another class, allowing code reuse and creating relationships between classes.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is inheritance in OOP?",
        "candidate_answer": "Inheritance allows one class to reuse or acquire properties and methods from another class.",
        "quality_label": "Good"
    },
    {
        "question": "What is inheritance in OOP?",
        "candidate_answer": "Inheritance allows classes to get features from another class.",
        "quality_label": "Average"
    },
    {
        "question": "What is inheritance in OOP?",
        "candidate_answer": "Inheritance is an OOP concept.",
        "quality_label": "Poor"
    },

    {
        "question": "What is encapsulation in OOP?",
        "candidate_answer": "Encapsulation is an object-oriented programming concept that combines data and methods inside a class and controls access to the internal data through appropriate interfaces.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is encapsulation in OOP?",
        "candidate_answer": "Encapsulation combines data and methods within a class and helps control access to the internal data.",
        "quality_label": "Good"
    },
    {
        "question": "What is encapsulation in OOP?",
        "candidate_answer": "Encapsulation keeps data and methods together inside a class.",
        "quality_label": "Average"
    },
    {
        "question": "What is encapsulation in OOP?",
        "candidate_answer": "It is an OOP concept.",
        "quality_label": "Poor"
    },

    # ---------------- DATA SCIENCE ----------------

    {
        "question": "What is data preprocessing?",
        "candidate_answer": "Data preprocessing is the process of cleaning and transforming raw data into a suitable format for analysis or machine learning. It can include handling missing values, removing duplicates, encoding categorical variables, and scaling numerical features.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is data preprocessing?",
        "candidate_answer": "Data preprocessing involves cleaning and transforming raw data before using it for analysis or machine learning.",
        "quality_label": "Good"
    },
    {
        "question": "What is data preprocessing?",
        "candidate_answer": "Data preprocessing means cleaning data before analysis.",
        "quality_label": "Average"
    },
    {
        "question": "What is data preprocessing?",
        "candidate_answer": "It is something done with data.",
        "quality_label": "Poor"
    },

    {
        "question": "What is exploratory data analysis?",
        "candidate_answer": "Exploratory data analysis is the process of examining and visualizing a dataset to understand distributions, patterns, relationships, trends, and unusual values before further analysis or modeling.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is exploratory data analysis?",
        "candidate_answer": "EDA involves examining and visualizing data to identify patterns, relationships, and unusual values.",
        "quality_label": "Good"
    },
    {
        "question": "What is exploratory data analysis?",
        "candidate_answer": "EDA is used to understand a dataset before building a model.",
        "quality_label": "Average"
    },
    {
        "question": "What is exploratory data analysis?",
        "candidate_answer": "It is analyzing data.",
        "quality_label": "Poor"
    },

    {
        "question": "What is feature engineering?",
        "candidate_answer": "Feature engineering is the process of creating, transforming, or selecting useful input features from raw data to improve the performance of a machine learning model.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is feature engineering?",
        "candidate_answer": "Feature engineering involves transforming or creating useful features from raw data to help a machine learning model perform better.",
        "quality_label": "Good"
    },
    {
        "question": "What is feature engineering?",
        "candidate_answer": "Feature engineering means preparing useful features from data for machine learning.",
        "quality_label": "Average"
    },
    {
        "question": "What is feature engineering?",
        "candidate_answer": "It is related to machine learning features.",
        "quality_label": "Poor"
    }
]

print("Total candidate answers:", len(candidate_answers))

Total candidate answers: 60


In [34]:
question_info = {
    item["question"]: item
    for item in question_bank
}

In [35]:
dataset = []

for answer in candidate_answers:

    question = answer["question"]

    info = question_info[question]

    dataset.append({
        "question": question,
        "expected_answer": info["expected_answer"],
        "candidate_answer": answer["candidate_answer"],
        "keywords": info["keywords"],
        "question_type": info["question_type"],
        "quality_label": answer["quality_label"]
    })

print("Total dataset samples:", len(dataset))

Total dataset samples: 60


In [36]:
df = pd.DataFrame(dataset)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (60, 6)


,question,expected_answer,candidate_answer,keywords,question_type,quality_label
0,What is Python?,Python is a high-level programming language kn...,Python is a high-level programming language kn...,"[python, high-level, programming language, syn...",Python,Excellent
1,What is Python?,Python is a high-level programming language kn...,Python is a high-level programming language wi...,"[python, high-level, programming language, syn...",Python,Good
2,What is Python?,Python is a high-level programming language kn...,Python is a programming language used to devel...,"[python, high-level, programming language, syn...",Python,Average
3,What is Python?,Python is a high-level programming language kn...,Python is a language used for programming.,"[python, high-level, programming language, syn...",Python,Poor
4,What is a list in Python?,A list is a mutable ordered collection in Pyth...,A list is a mutable and ordered collection in ...,"[list, mutable, ordered, collection, duplicate]",Python,Excellent


In [37]:
semantic_scores = []

for index, row in df.iterrows():

    expected_embedding = model.encode(row["expected_answer"])
    candidate_embedding = model.encode(row["candidate_answer"])

    similarity = cosine_similarity(
        [expected_embedding],
        [candidate_embedding]
    )

    semantic_scores.append(similarity[0][0])

df["semantic_score"] = semantic_scores

In [38]:
df["keyword_score"] = df.apply(
    lambda row: calculate_keyword_score(
        row["candidate_answer"],
        row["keywords"]
    ),
    axis=1
)
df["answer_length"] = df["candidate_answer"].apply(
    lambda answer: len(answer.split())
)

In [39]:
df[
    [
        "question",
        "quality_label",
        "semantic_score",
        "keyword_score",
        "answer_length"
    ]
]

,question,quality_label,semantic_score,keyword_score,answer_length
0,What is Python?,Excellent,0.928779,0.800000,28
1,What is Python?,Good,0.927963,0.800000,21
2,What is Python?,Average,0.812702,0.400000,11
3,What is Python?,Poor,0.843463,0.200000,7
4,What is a list in Python?,Excellent,0.971547,1.000000,24
5,What is a list in Python?,Good,0.904635,0.600000,18
6,What is a list in Python?,Average,0.781882,0.200000,7
7,What is a list in Python?,Poor,0.792838,0.200000,7
8,What is machine learning?,Excellent,0.981287,1.000000,28
9,What is machine learning?,Good,0.956307,1.000000,21


In [40]:
df["quality_label"].value_counts()

quality_label
Excellent    15
Good         15
Average      15
Poor         15
Name: count, dtype: int64

In [41]:
df[
    ["semantic_score", "keyword_score", "answer_length"]
].describe()

,semantic_score,keyword_score,answer_length
count,60.000000,60.000000,60.000000
mean,0.805793,0.562778,14.450000
std,0.183409,0.348421,8.116576
min,0.346152,0.000000,4.000000
25%,0.748827,0.200000,7.000000
50%,0.861881,0.600000,12.000000
75%,0.949375,1.000000,20.250000
max,0.995428,1.000000,35.000000


In [42]:
df.groupby("quality_label")[
    ["semantic_score", "keyword_score", "answer_length"]
].mean()

,semantic_score,keyword_score,answer_length
quality_label,,,
Average,0.774821,0.402222,9.200000
Excellent,0.974403,0.962222,25.866667
Good,0.862216,0.731111,16.733333
Poor,0.611733,0.155556,6.000000


In [43]:
df.isnull().sum()

question            0
expected_answer     0
candidate_answer    0
keywords            0
question_type       0
quality_label       0
semantic_score      0
keyword_score       0
answer_length       0
dtype: int64

In [44]:
quality_score_map = {
    "Poor": 25,
    "Average": 50,
    "Good": 75,
    "Excellent": 100
}

df["quality_score"] = df["quality_label"].map(quality_score_map)

df[["quality_label", "quality_score"]].drop_duplicates()

,quality_label,quality_score
0,Excellent,100
1,Good,75
2,Average,50
3,Poor,25


In [45]:
df["quality_score"].value_counts().sort_index()

quality_score
25     15
50     15
75     15
100    15
Name: count, dtype: int64

In [46]:
df.to_csv("interview_dataset_seed.csv", index=False)

In [47]:
import os
print(os.path.exists("interview_dataset_seed.csv"))

True


In [48]:
import random
import re

# Neutral wording variations.
# These replacements are applied across ALL quality levels,
# so we don't accidentally create label-specific patterns.

replacement_map = {
    "is a": ["is a", "is basically a", "can be described as a"],
    "is used to": ["is used to", "can be used to", "is commonly used to"],
    "allows": ["allows", "enables", "lets"],
    "used for": ["used for", "commonly used for", "often used for"],
    "process of": ["process of", "procedure of", "method of"],
    "helps": ["helps", "assists", "can help"],
    "data": ["data", "information"],
    "model": ["model", "machine learning model"],
    "programming": ["programming", "software development"],
    "important": ["important", "useful"],
    "identify": ["identify", "recognize"],
    "learn": ["learn", "discover patterns from"],
    "patterns": ["patterns", "relationships"],
    "new data": ["new data", "unseen data"]
}


def create_variation(text):
    """
    Creates a small wording variation while preserving
    the original meaning.
    """

    variation = text

    # Randomly replace some phrases
    for phrase, options in replacement_map.items():

        if phrase in variation.lower():

            replacement = random.choice(options)

            variation = re.sub(
                phrase,
                replacement,
                variation,
                count=1,
                flags=re.IGNORECASE
            )

    # Occasionally remove an unnecessary comma
    if random.random() < 0.3:
        variation = variation.replace(", ", " ")

    # Occasionally change the final punctuation
    if random.random() < 0.3:
        variation = variation.rstrip(".") + "."

    return variation

In [49]:
original = df.loc[0, "candidate_answer"]

print("Original:")
print(original)

print("\nVariations:")

for i in range(5):
    print(f"{i + 1}. {create_variation(original)}")

Original:
Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.

Variations:
1. Python can be described as a high-level programming language known for its simple and readable syntax. It is widely used in web development automation information science artificial intelligence and machine discover patterns froming.
2. Python is a high-level software development language known for its simple and readable syntax. It is widely used in web development, automation, information science, artificial intelligence, and machine discover relationships froming.
3. Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development automation data science artificial intelligence and machine learning.
4. Python is basically a high-level programming language known for its simple and readable syntax. It is wide

In [50]:
import random

neutral_prefixes = [
    "",
    "In simple terms, ",
    "Basically, ",
    "In general, ",
    "To put it simply, "
]

neutral_suffixes = [
    "",
    "",
    "",
    " This is an important concept.",
    " This is commonly used in practice."
]


def create_safe_variation(text):
    """
    Creates a conservative variation without replacing
    technical terms or changing the core meaning.
    """

    variation = text.strip()

    # Add a neutral prefix
    prefix = random.choice(neutral_prefixes)

    if prefix:
        variation = prefix + variation[0].lower() + variation[1:]

    # Add a neutral suffix occasionally
    suffix = random.choice(neutral_suffixes)

    if suffix and not variation.endswith("."):
        variation += "."

    variation += suffix

    return variation

In [51]:
original = df.loc[0, "candidate_answer"]

print("Original:")
print(original)

print("\nSafe variations:")

for i in range(5):
    print(f"{i + 1}. {create_safe_variation(original)}")

Original:
Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.

Safe variations:
1. In simple terms, python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning. This is an important concept.
2. To put it simply, python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning. This is an important concept.
3. In simple terms, python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.
4. Basically, python is a high-level programming language known

In [52]:
answer_templates = {
    "Excellent": [
        "{answer}",
        "{answer} It is widely used in real-world applications.",
        "In detail, {answer}",
        "{answer} It is an important concept in software development."
    ],

    "Good": [
        "{answer}",
        "In simple terms, {answer}",
        "{answer} It is commonly used in practice.",
        "Basically, {answer}"
    ],

    "Average": [
        "{answer}",
        "In short, {answer}",
        "Generally, {answer}",
        "{answer} This is a basic concept."
    ],

    "Poor": [
        "{answer}",
        "I have a basic understanding of this. {answer}",
        "I know that {answer}",
        "I have heard about this. {answer}"
    ]
}

In [53]:
for label in answer_templates:

    print("\n", label)

    answer = df[df["quality_label"] == label].iloc[0]["candidate_answer"]

    for template in answer_templates[label]:
        print("-", template.format(answer=answer))


 Excellent
- Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.
- Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning. It is widely used in real-world applications.
- In detail, Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.
- Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning. It is an important concept in software development.

 Good
- Python is a high-level programming language with simple syntax. It is easy

In [54]:
neutral_templates = [
    "{answer}",
    "In simple terms, {answer}",
    "Basically, {answer}",
    "In general, {answer}",
    "{answer} This is commonly used in practice."
]

In [55]:
augmented_data = []

for index, row in df.iterrows():

    for variation_id, template in enumerate(neutral_templates):

        new_answer = template.format(
            answer=row["candidate_answer"]
        )

        augmented_data.append({
            "seed_id": index,
            "variation_id": variation_id,
            "question": row["question"],
            "expected_answer": row["expected_answer"],
            "candidate_answer": new_answer,
            "keywords": row["keywords"],
            "question_type": row["question_type"],
            "quality_label": row["quality_label"],
            "quality_score": row["quality_score"]
        })


augmented_df = pd.DataFrame(augmented_data)

print("Augmented dataset shape:", augmented_df.shape)

Augmented dataset shape: (300, 9)


In [57]:
augmented_df["quality_label"].value_counts()


quality_label
Excellent    75
Good         75
Average      75
Poor         75
Name: count, dtype: int64

In [58]:
print(
    "Duplicate answers:",
    augmented_df["candidate_answer"].duplicated().sum()
)

Duplicate answers: 0


In [59]:
semantic_scores = []

for index, row in augmented_df.iterrows():

    expected_embedding = model.encode(row["expected_answer"])
    candidate_embedding = model.encode(row["candidate_answer"])

    similarity = cosine_similarity(
        [expected_embedding],
        [candidate_embedding]
    )

    semantic_scores.append(similarity[0][0])

augmented_df["semantic_score"] = semantic_scores

print("Semantic scores calculated!")

Semantic scores calculated!


In [60]:
augmented_df["keyword_score"] = augmented_df.apply(
    lambda row: calculate_keyword_score(
        row["candidate_answer"],
        row["keywords"]
    ),
    axis=1
)

print("Keyword scores calculated!")

Keyword scores calculated!


In [61]:
augmented_df["answer_length"] = augmented_df["candidate_answer"].apply(
    lambda answer: len(answer.split())
)

print("Answer lengths calculated!")

Answer lengths calculated!


In [62]:
augmented_df[
    [
        "question",
        "candidate_answer",
        "semantic_score",
        "keyword_score",
        "answer_length",
        "quality_label",
        "quality_score"
    ]
].head(10)

,question,candidate_answer,semantic_score,keyword_score,answer_length,quality_label,quality_score
0,What is Python?,Python is a high-level programming language kn...,0.928779,0.8,28,Excellent,100
1,What is Python?,"In simple terms, Python is a high-level progra...",0.919809,0.8,31,Excellent,100
2,What is Python?,"Basically, Python is a high-level programming ...",0.883642,0.8,29,Excellent,100
3,What is Python?,"In general, Python is a high-level programming...",0.913093,0.8,30,Excellent,100
4,What is Python?,Python is a high-level programming language kn...,0.913576,0.8,34,Excellent,100
5,What is Python?,Python is a high-level programming language wi...,0.927963,0.8,21,Good,75
6,What is Python?,"In simple terms, Python is a high-level progra...",0.895768,0.8,24,Good,75
7,What is Python?,"Basically, Python is a high-level programming ...",0.872873,0.8,22,Good,75
8,What is Python?,"In general, Python is a high-level programming...",0.899534,0.8,23,Good,75
9,What is Python?,Python is a high-level programming language wi...,0.906556,0.8,27,Good,75


In [63]:
augmented_df.groupby("quality_label")[
    ["semantic_score", "keyword_score", "answer_length"]
].mean().round(3)

,semantic_score,keyword_score,answer_length
quality_label,,,
Average,0.777,0.402,11.600
Excellent,0.958,0.962,28.267
Good,0.860,0.731,19.133
Poor,0.608,0.156,8.400


In [64]:
augmented_df[
    ["semantic_score", "keyword_score", "answer_length"]
].describe().round(3)

,semantic_score,keyword_score,answer_length
count,300.000,300.000,300.000
mean,0.801,0.563,16.850
std,0.179,0.346,8.322
min,0.334,0.000,4.000
25%,0.758,0.200,10.000
50%,0.860,0.600,15.000
75%,0.930,1.000,23.000
max,0.995,1.000,41.000


In [65]:
augmented_df.isnull().sum()

seed_id             0
variation_id        0
question            0
expected_answer     0
candidate_answer    0
keywords            0
question_type       0
quality_label       0
quality_score       0
semantic_score      0
keyword_score       0
answer_length       0
dtype: int64

In [66]:
X = augmented_df[
    [
        "semantic_score",
        "keyword_score",
        "answer_length"
    ]
]

y = augmented_df["quality_score"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (300, 3)
Target shape: (300,)


In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 240
Testing samples: 60


In [68]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model_rf.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [69]:
model_rf.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [70]:
y_pred_rf = model_rf.predict(X_test)

print("Actual values:")
print(y_test.values[:10])

print("\nPredicted values:")
print(y_pred_rf[:10])

Actual values:
[100  75  50  75  50  75  25  75  75  25]

Predicted values:
[100.    71.    50.    83.25  51.75  75.5   25.    81.75  79.25  25.25]


In [71]:
from sklearn.metrics import mean_absolute_error, r2_score

mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest MAE:", round(mae_rf, 3))
print("Random Forest R²:", round(r2_rf, 3))

Random Forest MAE: 5.883
Random Forest R²: 0.857


In [72]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42
    )
}

In [73]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE",
    ascending=True
).reset_index(drop=True)

results_df.round(3)

,Model,MAE,R2
0,Decision Tree,5.000,0.841
1,Random Forest,5.883,0.857
2,Gradient Boosting,6.159,0.871
3,Extra Trees,6.201,0.859
4,Linear Regression,8.131,0.876


In [74]:
from sklearn.model_selection import GroupShuffleSplit

X = augmented_df[
    [
        "semantic_score",
        "keyword_score",
        "answer_length"
    ]
]

y = augmented_df["quality_score"]

groups = augmented_df["seed_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 240
Testing samples: 60


In [75]:
print("Training seed IDs:")
print(sorted(augmented_df.iloc[train_idx]["seed_id"].unique()))

print("\nTesting seed IDs:")
print(sorted(augmented_df.iloc[test_idx]["seed_id"].unique()))

Training seed IDs:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(34), np.int64(35), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(47), np.int64(49), np.int64(51), np.int64(52), np.int64(53), np.int64(55), np.int64(56), np.int64(58), np.int64(59)]

Testing seed IDs:
[np.int64(0), np.int64(5), np.int64(12), np.int64(13), np.int64(33), np.int64(36), np.int64(45), np.int64(46), np.int64(48), np.int64(50), np.int64(54), np.int64(57)]


In [76]:
train_seeds = set(augmented_df.iloc[train_idx]["seed_id"])
test_seeds = set(augmented_df.iloc[test_idx]["seed_id"])

overlap = train_seeds.intersection(test_seeds)

print("Number of overlapping seed IDs:", len(overlap))
print("Overlapping IDs:", overlap)

Number of overlapping seed IDs: 0
Overlapping IDs: set()


In [77]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE",
    ascending=True
).reset_index(drop=True)

results_df.round(3)

,Model,MAE,R2
0,Decision Tree,6.667,0.537
1,Random Forest,7.071,0.719
2,Gradient Boosting,7.450,0.712
3,Linear Regression,7.827,0.740
4,Extra Trees,8.854,0.617


In [78]:
augmented_df["expected_length"] = augmented_df["expected_answer"].apply(
    lambda answer: len(answer.split())
)

augmented_df["length_ratio"] = (
    augmented_df["answer_length"] /
    augmented_df["expected_length"]
)

augmented_df[
    [
        "answer_length",
        "expected_length",
        "length_ratio",
        "quality_label"
    ]
].head(10)

,answer_length,expected_length,length_ratio,quality_label
0,28,13,2.153846,Excellent
1,31,13,2.384615,Excellent
2,29,13,2.230769,Excellent
3,30,13,2.307692,Excellent
4,34,13,2.615385,Excellent
5,21,13,1.615385,Good
6,24,13,1.846154,Good
7,22,13,1.692308,Good
8,23,13,1.769231,Good
9,27,13,2.076923,Good


In [79]:
def count_matched_keywords(row):
    answer = row["candidate_answer"].lower()
    keywords = row["keywords"]

    return sum(
        1 for keyword in keywords
        if keyword.lower() in answer
    )


augmented_df["matched_keywords"] = augmented_df.apply(
    count_matched_keywords,
    axis=1
)

augmented_df["total_keywords"] = augmented_df["keywords"].apply(
    len
)

augmented_df[
    [
        "matched_keywords",
        "total_keywords",
        "keyword_score",
        "quality_label"
    ]
].head(10)

,matched_keywords,total_keywords,keyword_score,quality_label
0,4,5,0.8,Excellent
1,4,5,0.8,Excellent
2,4,5,0.8,Excellent
3,4,5,0.8,Excellent
4,4,5,0.8,Excellent
5,4,5,0.8,Good
6,4,5,0.8,Good
7,4,5,0.8,Good
8,4,5,0.8,Good
9,4,5,0.8,Good


In [80]:
import re

def count_sentences(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return len(sentences)


augmented_df["sentence_count"] = augmented_df["candidate_answer"].apply(
    count_sentences
)

augmented_df["avg_sentence_length"] = (
    augmented_df["answer_length"] /
    augmented_df["sentence_count"]
)

augmented_df[
    [
        "answer_length",
        "sentence_count",
        "avg_sentence_length",
        "quality_label"
    ]
].head(10)

,answer_length,sentence_count,avg_sentence_length,quality_label
0,28,2,14.000000,Excellent
1,31,2,15.500000,Excellent
2,29,2,14.500000,Excellent
3,30,2,15.000000,Excellent
4,34,3,11.333333,Excellent
5,21,2,10.500000,Good
6,24,2,12.000000,Good
7,22,2,11.000000,Good
8,23,2,11.500000,Good
9,27,3,9.000000,Good


In [81]:
def unique_word_ratio(text):
    words = text.lower().split()
    
    if len(words) == 0:
        return 0
    
    unique_words = set(words)
    
    return len(unique_words) / len(words)


augmented_df["unique_word_ratio"] = augmented_df["candidate_answer"].apply(
    unique_word_ratio
)

augmented_df[
    [
        "answer_length",
        "unique_word_ratio",
        "quality_label"
    ]
].head(10)

,answer_length,unique_word_ratio,quality_label
0,28,0.928571,Excellent
1,31,0.870968,Excellent
2,29,0.931034,Excellent
3,30,0.900000,Excellent
4,34,0.852941,Excellent
5,21,0.904762,Good
6,24,0.833333,Good
7,22,0.909091,Good
8,23,0.869565,Good
9,27,0.814815,Good


In [82]:
augmented_df[
    [
        "semantic_score",
        "keyword_score",
        "answer_length",
        "expected_length",
        "length_ratio",
        "matched_keywords",
        "total_keywords",
        "sentence_count",
        "avg_sentence_length",
        "unique_word_ratio",
        "quality_label",
        "quality_score"
    ]
].head(10)

,semantic_score,keyword_score,answer_length,expected_length,length_ratio,matched_keywords,total_keywords,sentence_count,avg_sentence_length,unique_word_ratio,quality_label,quality_score
0,0.928779,0.8,28,13,2.153846,4,5,2,14.000000,0.928571,Excellent,100
1,0.919809,0.8,31,13,2.384615,4,5,2,15.500000,0.870968,Excellent,100
2,0.883642,0.8,29,13,2.230769,4,5,2,14.500000,0.931034,Excellent,100
3,0.913093,0.8,30,13,2.307692,4,5,2,15.000000,0.900000,Excellent,100
4,0.913576,0.8,34,13,2.615385,4,5,3,11.333333,0.852941,Excellent,100
5,0.927963,0.8,21,13,1.615385,4,5,2,10.500000,0.904762,Good,75
6,0.895768,0.8,24,13,1.846154,4,5,2,12.000000,0.833333,Good,75
7,0.872873,0.8,22,13,1.692308,4,5,2,11.000000,0.909091,Good,75
8,0.899534,0.8,23,13,1.769231,4,5,2,11.500000,0.869565,Good,75
9,0.906556,0.8,27,13,2.076923,4,5,3,9.000000,0.814815,Good,75


In [83]:
feature_columns = [
    "semantic_score",
    "keyword_score",
    "answer_length",
    "expected_length",
    "length_ratio",
    "matched_keywords",
    "total_keywords",
    "sentence_count",
    "avg_sentence_length",
    "unique_word_ratio"
]

augmented_df.groupby("quality_label")[feature_columns].mean().round(3)

,semantic_score,keyword_score,answer_length,expected_length,length_ratio,matched_keywords,total_keywords,sentence_count,avg_sentence_length,unique_word_ratio
quality_label,,,,,,,,,,
Average,0.777,0.402,11.600,18.533,0.636,2.067,5.2,1.200,10.08,0.955
Excellent,0.958,0.962,28.267,18.533,1.547,5.000,5.2,1.467,21.40,0.915
Good,0.860,0.731,19.133,18.533,1.050,3.800,5.2,1.267,16.20,0.950
Poor,0.608,0.156,8.400,18.533,0.459,0.800,5.2,1.200,7.20,0.972


In [84]:
feature_columns = [
    "semantic_score",
    "keyword_score",
    "answer_length",
    "length_ratio",
    "matched_keywords",
    "sentence_count",
    "avg_sentence_length",
    "unique_word_ratio"
]

X = augmented_df[feature_columns]

y = augmented_df["quality_score"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (300, 8)
Target shape: (300,)


In [85]:
from sklearn.model_selection import GroupShuffleSplit

groups = augmented_df["seed_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 240
Testing samples: 60


In [86]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42
    )
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE"
).reset_index(drop=True)

results_df

,Model,MAE,R2
0,Gradient Boosting,4.844023,0.877388
1,Random Forest,5.258333,0.849465
2,Linear Regression,6.087605,0.845412
3,Extra Trees,6.104167,0.785923
4,Decision Tree,6.666667,0.537349


In [87]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone

group_kfold = GroupKFold(n_splits=5)

cv_results = []

for name, model in models.items():

    fold_mae = []
    fold_r2 = []

    for train_idx, val_idx in group_kfold.split(
        X, y, groups=groups
    ):
        model_copy = clone(model)

        X_train_cv = X.iloc[train_idx]
        X_val_cv = X.iloc[val_idx]

        y_train_cv = y.iloc[train_idx]
        y_val_cv = y.iloc[val_idx]

        model_copy.fit(X_train_cv, y_train_cv)

        predictions = model_copy.predict(X_val_cv)

        fold_mae.append(
            mean_absolute_error(y_val_cv, predictions)
        )

        fold_r2.append(
            r2_score(y_val_cv, predictions)
        )

    cv_results.append({
        "Model": name,
        "Mean MAE": sum(fold_mae) / len(fold_mae),
        "Mean R2": sum(fold_r2) / len(fold_r2)
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="Mean MAE"
).reset_index(drop=True)

cv_results_df

,Model,Mean MAE,Mean R2
0,Random Forest,6.050833,0.859493
1,Gradient Boosting,6.274356,0.881771
2,Extra Trees,6.402500,0.870412
3,Decision Tree,6.500000,0.781333
4,Linear Regression,6.989381,0.898643


In [88]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X, y)

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance

,Feature,Importance
0,answer_length,0.712386
1,length_ratio,0.112022
2,avg_sentence_length,0.075963
3,semantic_score,0.053183
4,keyword_score,0.035167
5,unique_word_ratio,0.005548
6,matched_keywords,0.005258
7,sentence_count,0.000473


In [89]:
reduced_features = [
    "semantic_score",
    "keyword_score",
    "length_ratio",
    "matched_keywords",
    "sentence_count",
    "avg_sentence_length",
    "unique_word_ratio"
]

X_reduced = augmented_df[reduced_features]

cv_results_reduced = []

for name, model in models.items():

    fold_mae = []
    fold_r2 = []

    for train_idx, val_idx in group_kfold.split(
        X_reduced,
        y,
        groups=groups
    ):
        model_copy = clone(model)

        X_train_cv = X_reduced.iloc[train_idx]
        X_val_cv = X_reduced.iloc[val_idx]

        y_train_cv = y.iloc[train_idx]
        y_val_cv = y.iloc[val_idx]

        model_copy.fit(X_train_cv, y_train_cv)

        predictions = model_copy.predict(X_val_cv)

        fold_mae.append(
            mean_absolute_error(y_val_cv, predictions)
        )

        fold_r2.append(
            r2_score(y_val_cv, predictions)
        )

    cv_results_reduced.append({
        "Model": name,
        "Mean MAE": sum(fold_mae) / len(fold_mae),
        "Mean R2": sum(fold_r2) / len(fold_r2)
    })

reduced_results_df = pd.DataFrame(cv_results_reduced)

reduced_results_df.sort_values(
    by="Mean MAE"
).reset_index(drop=True)

,Model,Mean MAE,Mean R2
0,Decision Tree,5.750000,0.816000
1,Random Forest,6.532500,0.843822
2,Gradient Boosting,6.726662,0.862582
3,Extra Trees,7.082083,0.847723
4,Linear Regression,7.259583,0.892423


In [90]:
cv_stability_results = []

for name, model in models.items():

    fold_mae = []
    fold_r2 = []

    for train_idx, val_idx in group_kfold.split(
        X_reduced,
        y,
        groups=groups
    ):
        model_copy = clone(model)

        X_train_cv = X_reduced.iloc[train_idx]
        X_val_cv = X_reduced.iloc[val_idx]

        y_train_cv = y.iloc[train_idx]
        y_val_cv = y.iloc[val_idx]

        model_copy.fit(X_train_cv, y_train_cv)

        predictions = model_copy.predict(X_val_cv)

        fold_mae.append(
            mean_absolute_error(y_val_cv, predictions)
        )

        fold_r2.append(
            r2_score(y_val_cv, predictions)
        )

    cv_stability_results.append({
        "Model": name,
        "Mean MAE": sum(fold_mae) / len(fold_mae),
        "MAE Std": pd.Series(fold_mae).std(),
        "Mean R2": sum(fold_r2) / len(fold_r2),
        "R2 Std": pd.Series(fold_r2).std()
    })

stability_df = pd.DataFrame(cv_stability_results)

stability_df.sort_values(
    by="Mean MAE"
).reset_index(drop=True)

,Model,Mean MAE,MAE Std,Mean R2,R2 Std
0,Decision Tree,5.750000,3.338538,0.816000,0.106833
1,Random Forest,6.532500,2.602649,0.843822,0.081911
2,Gradient Boosting,6.726662,1.936654,0.862582,0.065838
3,Extra Trees,7.082083,2.635709,0.847723,0.075622
4,Linear Regression,7.259583,1.212126,0.892423,0.038518


In [91]:
from sklearn.model_selection import GroupShuffleSplit

final_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=123
)

dev_idx, final_test_idx = next(
    final_split.split(X_reduced, y, groups=groups)
)

X_dev = X_reduced.iloc[dev_idx]
X_final_test = X_reduced.iloc[final_test_idx]

y_dev = y.iloc[dev_idx]
y_final_test = y.iloc[final_test_idx]

print("Development samples:", len(X_dev))
print("Final test samples:", len(X_final_test))

Development samples: 240
Final test samples: 60


In [92]:
group_kfold_dev = GroupKFold(n_splits=5)

dev_results = []

dev_groups = groups.iloc[dev_idx]

for name, model in models.items():

    fold_mae = []
    fold_r2 = []

    for train_fold_idx, val_fold_idx in group_kfold_dev.split(
        X_dev,
        y_dev,
        groups=dev_groups
    ):
        model_copy = clone(model)

        X_train_fold = X_dev.iloc[train_fold_idx]
        X_val_fold = X_dev.iloc[val_fold_idx]

        y_train_fold = y_dev.iloc[train_fold_idx]
        y_val_fold = y_dev.iloc[val_fold_idx]

        model_copy.fit(X_train_fold, y_train_fold)

        predictions = model_copy.predict(X_val_fold)

        fold_mae.append(
            mean_absolute_error(y_val_fold, predictions)
        )

        fold_r2.append(
            r2_score(y_val_fold, predictions)
        )

    dev_results.append({
        "Model": name,
        "Mean MAE": sum(fold_mae) / len(fold_mae),
        "Mean R2": sum(fold_r2) / len(fold_r2)
    })

dev_results_df = pd.DataFrame(dev_results)

dev_results_df.sort_values(
    by="Mean MAE"
).reset_index(drop=True)

,Model,Mean MAE,Mean R2
0,Linear Regression,7.886118,0.810822
1,Gradient Boosting,7.952480,0.747427
2,Random Forest,8.062000,0.726025
3,Extra Trees,8.484556,0.721186
4,Decision Tree,8.933333,0.571377


In [93]:
final_model = LinearRegression()

final_model.fit(X_dev, y_dev)

final_predictions = final_model.predict(X_final_test)

final_mae = mean_absolute_error(
    y_final_test,
    final_predictions
)

final_r2 = r2_score(
    y_final_test,
    final_predictions
)

print("Final Model: Linear Regression")
print("Final MAE:", round(final_mae, 3))
print("Final R²:", round(final_r2, 3))

Final Model: Linear Regression
Final MAE: 6.226
Final R²: 0.944


In [94]:
prediction_results = pd.DataFrame({
    "Actual Score": y_final_test.values,
    "Predicted Score": final_predictions
})

prediction_results["Error"] = (
    prediction_results["Predicted Score"] -
    prediction_results["Actual Score"]
).round(2)

prediction_results.head(15)

,Actual Score,Predicted Score,Error
0,75,70.889644,-4.11
1,75,77.884512,2.88
2,75,73.040655,-1.96
3,75,75.567778,0.57
4,75,67.498082,-7.50
5,25,28.792704,3.79
6,25,36.026668,11.03
7,25,31.035686,6.04
8,25,33.355102,8.36
9,25,35.858462,10.86


In [95]:
seed_df = df.copy()

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nQuestions:", seed_df["question"].nunique())

Seed dataset shape: (60, 10)

Quality distribution:
quality_label
Excellent    15
Good         15
Average      15
Poor         15
Name: count, dtype: int64

Questions: 15


In [96]:
seed_df = seed_df.reset_index(drop=True)

seed_df["seed_id"] = range(1, len(seed_df) + 1)

seed_df[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label"
    ]
].head(10)

,seed_id,question,candidate_answer,quality_label
0,1,What is Python?,Python is a high-level programming language kn...,Excellent
1,2,What is Python?,Python is a high-level programming language wi...,Good
2,3,What is Python?,Python is a programming language used to devel...,Average
3,4,What is Python?,Python is a language used for programming.,Poor
4,5,What is a list in Python?,A list is a mutable and ordered collection in ...,Excellent
5,6,What is a list in Python?,A Python list is an ordered collection that ca...,Good
6,7,What is a list in Python?,A list stores multiple values in Python.,Average
7,8,What is a list in Python?,A list is something used in Python.,Poor
8,9,What is machine learning?,Machine learning is a branch of artificial int...,Excellent
9,10,What is machine learning?,Machine learning is a part of artificial intel...,Good


In [97]:
additional_questions = [
    {
        "question": "What is NumPy?",
        "expected_answer": "NumPy is a Python library used for numerical computing. It provides multidimensional arrays and efficient mathematical operations on data.",
        "keywords": ["NumPy", "numerical", "arrays", "mathematical"],
        "question_type": "Data Science"
    },
    {
        "question": "What is Pandas?",
        "expected_answer": "Pandas is a Python library used for data manipulation and analysis. It provides data structures such as Series and DataFrame for working with structured data.",
        "keywords": ["Pandas", "data manipulation", "DataFrame", "Series"],
        "question_type": "Data Science"
    },
    {
        "question": "What is mean in statistics?",
        "expected_answer": "Mean is the average of a set of numerical values. It is calculated by adding all the values and dividing the sum by the number of values.",
        "keywords": ["mean", "average", "sum", "values"],
        "question_type": "Statistics"
    },
    {
        "question": "What is a stack in data structures?",
        "expected_answer": "A stack is a linear data structure that follows the Last In First Out principle. Elements are added and removed from the top of the stack.",
        "keywords": ["stack", "LIFO", "data structure", "top"],
        "question_type": "Data Structures"
    },
    {
        "question": "What is binary search?",
        "expected_answer": "Binary search is an efficient searching algorithm that works on sorted data by repeatedly dividing the search space into two halves.",
        "keywords": ["binary search", "sorted", "algorithm", "halves"],
        "question_type": "Algorithms"
    }
]

print("New questions:", len(additional_questions))

New questions: 5


In [98]:
new_seed_answers = [
    # NumPy
    {
        "question": "What is NumPy?",
        "candidate_answer": "NumPy is a Python library used for numerical computing. It provides multidimensional arrays and efficient mathematical operations on data.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is NumPy?",
        "candidate_answer": "NumPy is a Python library for numerical computing that provides arrays and mathematical operations.",
        "quality_label": "Good"
    },
    {
        "question": "What is NumPy?",
        "candidate_answer": "NumPy is used to work with numerical data and arrays in Python.",
        "quality_label": "Average"
    },
    {
        "question": "What is NumPy?",
        "candidate_answer": "NumPy is a Python library.",
        "quality_label": "Poor"
    },

    # Pandas
    {
        "question": "What is Pandas?",
        "candidate_answer": "Pandas is a Python library used for data manipulation and analysis. It provides data structures such as Series and DataFrame for working with structured data.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is Pandas?",
        "candidate_answer": "Pandas is a Python library used for manipulating and analyzing structured data using DataFrames and Series.",
        "quality_label": "Good"
    },
    {
        "question": "What is Pandas?",
        "candidate_answer": "Pandas is commonly used to work with and analyze data in Python.",
        "quality_label": "Average"
    },
    {
        "question": "What is Pandas?",
        "candidate_answer": "Pandas is a Python library for data.",
        "quality_label": "Poor"
    },

    # Mean
    {
        "question": "What is mean in statistics?",
        "candidate_answer": "Mean is the average of a set of numerical values. It is calculated by adding all the values and dividing the sum by the number of values.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is mean in statistics?",
        "candidate_answer": "Mean is the average value of a dataset, calculated by dividing the sum of the values by the number of values.",
        "quality_label": "Good"
    },
    {
        "question": "What is mean in statistics?",
        "candidate_answer": "Mean is used to find the average of numerical values.",
        "quality_label": "Average"
    },
    {
        "question": "What is mean in statistics?",
        "candidate_answer": "Mean is an average.",
        "quality_label": "Poor"
    },

    # Stack
    {
        "question": "What is a stack in data structures?",
        "candidate_answer": "A stack is a linear data structure that follows the Last In First Out principle. Elements are added and removed from the top of the stack.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a stack in data structures?",
        "candidate_answer": "A stack is a data structure that follows LIFO, where elements are inserted and removed from the top.",
        "quality_label": "Good"
    },
    {
        "question": "What is a stack in data structures?",
        "candidate_answer": "A stack stores elements and follows the LIFO principle.",
        "quality_label": "Average"
    },
    {
        "question": "What is a stack in data structures?",
        "candidate_answer": "A stack is used to store data.",
        "quality_label": "Poor"
    },

    # Binary Search
    {
        "question": "What is binary search?",
        "candidate_answer": "Binary search is an efficient searching algorithm that works on sorted data by repeatedly dividing the search space into two halves.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is binary search?",
        "candidate_answer": "Binary search searches sorted data by repeatedly dividing the search range into two halves.",
        "quality_label": "Good"
    },
    {
        "question": "What is binary search?",
        "candidate_answer": "Binary search is an algorithm used to find an element in sorted data.",
        "quality_label": "Average"
    },
    {
        "question": "What is binary search?",
        "candidate_answer": "Binary search is a searching algorithm.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(new_seed_answers))

New seed answers: 20


In [99]:
new_seed_df = pd.DataFrame(new_seed_answers)

# Add missing columns so both datasets have the same structure
new_seed_df["expected_answer"] = new_seed_df["question"].map(
    {
        item["question"]: item["expected_answer"]
        for item in additional_questions
    }
)

new_seed_df["keywords"] = new_seed_df["question"].map(
    {
        item["question"]: item["keywords"]
        for item in additional_questions
    }
)

new_seed_df["question_type"] = new_seed_df["question"].map(
    {
        item["question"]: item["question_type"]
        for item in additional_questions
    }
)

# Add quality scores
new_seed_df["quality_score"] = new_seed_df["quality_label"].map(
    quality_score_map
)

# Combine old and new seeds
seed_df = pd.concat(
    [
        seed_df,
        new_seed_df[
            [
                "question",
                "expected_answer",
                "candidate_answer",
                "keywords",
                "question_type",
                "quality_label",
                "quality_score"
            ]
        ]
    ],
    ignore_index=True
)

# Re-create seed IDs
seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("New seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

New seed dataset shape: (80, 11)

Quality distribution:
quality_label
Excellent    20
Good         20
Average      20
Poor         20
Name: count, dtype: int64

Number of questions: 20


In [100]:
additional_questions_2 = [
    {
        "question": "What is a tuple in Python?",
        "expected_answer": "A tuple is an ordered and immutable collection in Python. It can store multiple values and is commonly used when the data should not be changed.",
        "keywords": ["tuple", "ordered", "immutable", "collection"],
        "question_type": "Python"
    },
    {
        "question": "What is a dictionary in Python?",
        "expected_answer": "A dictionary is a mutable data structure in Python that stores data as key-value pairs. Keys are used to access their corresponding values.",
        "keywords": ["dictionary", "key-value", "keys", "values"],
        "question_type": "Python"
    },
    {
        "question": "What is a queue in data structures?",
        "expected_answer": "A queue is a linear data structure that follows the First In First Out principle. Elements are inserted at the rear and removed from the front.",
        "keywords": ["queue", "FIFO", "rear", "front"],
        "question_type": "Data Structures"
    },
    {
        "question": "What is a primary key in SQL?",
        "expected_answer": "A primary key is a column or set of columns that uniquely identifies each row in a database table. It cannot contain duplicate or null values.",
        "keywords": ["primary key", "unique", "row", "null"],
        "question_type": "SQL"
    },
    {
        "question": "What is polymorphism in OOP?",
        "expected_answer": "Polymorphism is an OOP concept that allows the same interface or method name to perform different behaviors depending on the object or context.",
        "keywords": ["polymorphism", "OOP", "method", "different behavior"],
        "question_type": "OOP"
    }
]

print("New questions:", len(additional_questions_2))

New questions: 5


In [101]:
new_seed_answers_2 = [
    # Tuple
    {
        "question": "What is a tuple in Python?",
        "candidate_answer": "A tuple is an ordered and immutable collection in Python. It can store multiple values and is useful when the data should not be changed.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a tuple in Python?",
        "candidate_answer": "A tuple is an ordered collection in Python that cannot be modified after it is created.",
        "quality_label": "Good"
    },
    {
        "question": "What is a tuple in Python?",
        "candidate_answer": "A tuple is used to store multiple values and is immutable.",
        "quality_label": "Average"
    },
    {
        "question": "What is a tuple in Python?",
        "candidate_answer": "A tuple is a Python data type.",
        "quality_label": "Poor"
    },

    # Dictionary
    {
        "question": "What is a dictionary in Python?",
        "candidate_answer": "A dictionary is a mutable data structure in Python that stores data as key-value pairs. Keys are used to access their corresponding values.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a dictionary in Python?",
        "candidate_answer": "A dictionary stores data as key-value pairs, where keys are used to access values.",
        "quality_label": "Good"
    },
    {
        "question": "What is a dictionary in Python?",
        "candidate_answer": "A dictionary is used to store data using keys and values.",
        "quality_label": "Average"
    },
    {
        "question": "What is a dictionary in Python?",
        "candidate_answer": "A dictionary is used to store data.",
        "quality_label": "Poor"
    },

    # Queue
    {
        "question": "What is a queue in data structures?",
        "candidate_answer": "A queue is a linear data structure that follows the First In First Out principle. Elements are inserted at the rear and removed from the front.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a queue in data structures?",
        "candidate_answer": "A queue is a data structure that follows FIFO, where elements are added at the rear and removed from the front.",
        "quality_label": "Good"
    },
    {
        "question": "What is a queue in data structures?",
        "candidate_answer": "A queue stores elements and follows the FIFO principle.",
        "quality_label": "Average"
    },
    {
        "question": "What is a queue in data structures?",
        "candidate_answer": "A queue is used to store data.",
        "quality_label": "Poor"
    },

    # Primary Key
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key is a column or set of columns that uniquely identifies each row in a database table. It cannot contain duplicate or null values.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key uniquely identifies each row in a database table and cannot contain duplicate values.",
        "quality_label": "Good"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key is used to uniquely identify records in a table.",
        "quality_label": "Average"
    },
    {
        "question": "What is a primary key in SQL?",
        "candidate_answer": "A primary key identifies data in a table.",
        "quality_label": "Poor"
    },

    # Polymorphism
    {
        "question": "What is polymorphism in OOP?",
        "candidate_answer": "Polymorphism is an OOP concept that allows the same interface or method name to perform different behaviors depending on the object or context.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is polymorphism in OOP?",
        "candidate_answer": "Polymorphism allows the same method or interface to behave differently depending on the object or context.",
        "quality_label": "Good"
    },
    {
        "question": "What is polymorphism in OOP?",
        "candidate_answer": "Polymorphism means that the same method can have different behavior.",
        "quality_label": "Average"
    },
    {
        "question": "What is polymorphism in OOP?",
        "candidate_answer": "Polymorphism is an OOP concept.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(new_seed_answers_2))

New seed answers: 20


In [102]:
new_seed_df_2 = pd.DataFrame(new_seed_answers_2)

# Question metadata
expected_map_2 = {
    item["question"]: item["expected_answer"]
    for item in additional_questions_2
}

keyword_map_2 = {
    item["question"]: item["keywords"]
    for item in additional_questions_2
}

type_map_2 = {
    item["question"]: item["question_type"]
    for item in additional_questions_2
}

new_seed_df_2["expected_answer"] = new_seed_df_2["question"].map(
    expected_map_2
)

new_seed_df_2["keywords"] = new_seed_df_2["question"].map(
    keyword_map_2
)

new_seed_df_2["question_type"] = new_seed_df_2["question"].map(
    type_map_2
)

new_seed_df_2["quality_score"] = new_seed_df_2["quality_label"].map(
    quality_score_map
)

# Keep only the columns we need
new_seed_df_2 = new_seed_df_2[
    [
        "question",
        "expected_answer",
        "candidate_answer",
        "keywords",
        "question_type",
        "quality_label",
        "quality_score"
    ]
]

# Merge with existing seeds
seed_df = pd.concat(
    [seed_df, new_seed_df_2],
    ignore_index=True
)

# Re-create unique seed IDs
seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (100, 11)

Quality distribution:
quality_label
Excellent    25
Good         25
Average      25
Poor         25
Name: count, dtype: int64

Number of questions: 24


In [103]:
duplicate_questions = (
    seed_df["question"]
    .value_counts()
    .reset_index()
)

duplicate_questions.columns = ["question", "count"]

duplicate_questions[
    duplicate_questions["count"] > 4
]

,question,count
0,What is a primary key in SQL?,8


In [104]:
seed_df = seed_df.drop_duplicates(
    subset=["question", "quality_label"],
    keep="first"
).reset_index(drop=True)

seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (96, 11)

Quality distribution:
quality_label
Excellent    24
Good         24
Average      24
Poor         24
Name: count, dtype: int64

Number of questions: 24


In [105]:
additional_question_3 = {
    "question": "What is a JOIN in SQL?",
    "expected_answer": "A JOIN in SQL is used to combine rows from two or more tables based on a related column between them.",
    "keywords": ["JOIN", "SQL", "tables", "related", "column"],
    "question_type": "SQL"
}

print(additional_question_3["question"])

What is a JOIN in SQL?


In [106]:
join_seed_answers = [
    {
        "question": "What is a JOIN in SQL?",
        "candidate_answer": "A JOIN in SQL combines rows from two or more tables using a related column. Common types include INNER JOIN, LEFT JOIN, RIGHT JOIN, and FULL JOIN.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a JOIN in SQL?",
        "candidate_answer": "A JOIN combines data from multiple tables based on a related column.",
        "quality_label": "Good"
    },
    {
        "question": "What is a JOIN in SQL?",
        "candidate_answer": "A JOIN is used to get data from different tables together.",
        "quality_label": "Average"
    },
    {
        "question": "What is a JOIN in SQL?",
        "candidate_answer": "A JOIN connects tables in SQL.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(join_seed_answers))

New seed answers: 4


In [107]:
join_df = pd.DataFrame(join_seed_answers)

join_df["expected_answer"] = additional_question_3["expected_answer"]
join_df["keywords"] = [additional_question_3["keywords"]] * len(join_df)
join_df["question_type"] = additional_question_3["question_type"]

join_df["quality_score"] = join_df["quality_label"].map(
    quality_score_map
)

join_df = join_df[
    [
        "question",
        "expected_answer",
        "candidate_answer",
        "keywords",
        "question_type",
        "quality_label",
        "quality_score"
    ]
]

seed_df = pd.concat(
    [seed_df, join_df],
    ignore_index=True
)

seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (100, 11)

Quality distribution:
quality_label
Excellent    25
Good         25
Average      25
Poor         25
Name: count, dtype: int64

Number of questions: 25


In [108]:
additional_questions_4 = [
    {
        "question": "What is a Python function?",
        "expected_answer": "A function is a reusable block of Python code designed to perform a specific task. It can accept parameters and return a result.",
        "keywords": ["function", "reusable", "parameters", "return"],
        "question_type": "Python"
    },
    {
        "question": "What is overfitting in machine learning?",
        "expected_answer": "Overfitting occurs when a machine learning model learns the training data too closely, including noise, and performs poorly on unseen data.",
        "keywords": ["overfitting", "training data", "noise", "unseen data"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is train-test splitting?",
        "expected_answer": "Train-test splitting divides a dataset into training and testing subsets. The training data is used to learn the model, while the testing data evaluates its performance on unseen data.",
        "keywords": ["train", "test", "training data", "testing data", "unseen data"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is a DataFrame in Pandas?",
        "expected_answer": "A DataFrame is a two-dimensional labeled data structure in Pandas that organizes data into rows and columns and supports data manipulation and analysis.",
        "keywords": ["DataFrame", "Pandas", "rows", "columns", "data"],
        "question_type": "Data Science"
    },
    {
        "question": "What is normalization in machine learning?",
        "expected_answer": "Normalization is a feature scaling technique that transforms numerical values to a common range, often between zero and one, so features can be compared more effectively.",
        "keywords": ["normalization", "scaling", "features", "range", "zero", "one"],
        "question_type": "Machine Learning"
    }
]

# Check that every question is new
existing_questions = set(seed_df["question"])

new_questions = [
    item for item in additional_questions_4
    if item["question"] not in existing_questions
]

print("New unique questions:", len(new_questions))

New unique questions: 4


In [109]:
new_seed_answers_4 = [
    # Python Function
    {
        "question": "What is a Python function?",
        "candidate_answer": "A function is a reusable block of Python code designed to perform a specific task. It can accept parameters and return a result.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a Python function?",
        "candidate_answer": "A Python function is a reusable block of code that performs a specific task and can accept parameters and return a result.",
        "quality_label": "Good"
    },
    {
        "question": "What is a Python function?",
        "candidate_answer": "A function is used to perform a particular task in Python.",
        "quality_label": "Average"
    },
    {
        "question": "What is a Python function?",
        "candidate_answer": "A function is a block of code.",
        "quality_label": "Poor"
    },

    # Train-Test Splitting
    {
        "question": "What is train-test splitting?",
        "candidate_answer": "Train-test splitting divides a dataset into training and testing subsets. The training data is used to learn the model, while the testing data evaluates its performance on unseen data.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is train-test splitting?",
        "candidate_answer": "Train-test splitting divides data into training and testing sets. The training set is used to build the model and the test set evaluates it.",
        "quality_label": "Good"
    },
    {
        "question": "What is train-test splitting?",
        "candidate_answer": "It separates a dataset into training data and testing data.",
        "quality_label": "Average"
    },
    {
        "question": "What is train-test splitting?",
        "candidate_answer": "It splits the data.",
        "quality_label": "Poor"
    },

    # DataFrame
    {
        "question": "What is a DataFrame in Pandas?",
        "candidate_answer": "A DataFrame is a two-dimensional labeled data structure in Pandas that organizes data into rows and columns and supports data manipulation and analysis.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a DataFrame in Pandas?",
        "candidate_answer": "A DataFrame is a two-dimensional structure in Pandas that stores data in rows and columns.",
        "quality_label": "Good"
    },
    {
        "question": "What is a DataFrame in Pandas?",
        "candidate_answer": "A DataFrame is used to store and work with data in Pandas.",
        "quality_label": "Average"
    },
    {
        "question": "What is a DataFrame in Pandas?",
        "candidate_answer": "A DataFrame is a Pandas data structure.",
        "quality_label": "Poor"
    },

    # Normalization
    {
        "question": "What is normalization in machine learning?",
        "candidate_answer": "Normalization is a feature scaling technique that transforms numerical values to a common range, often between zero and one, so features can be compared more effectively.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is normalization in machine learning?",
        "candidate_answer": "Normalization scales numerical features to a common range, often between zero and one.",
        "quality_label": "Good"
    },
    {
        "question": "What is normalization in machine learning?",
        "candidate_answer": "Normalization is used to scale numerical data.",
        "quality_label": "Average"
    },
    {
        "question": "What is normalization in machine learning?",
        "candidate_answer": "Normalization changes the scale of data.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(new_seed_answers_4))

New seed answers: 16


In [110]:
new_seed_df_4 = pd.DataFrame(new_seed_answers_4)

# Metadata for the 4 new questions
expected_map_4 = {
    item["question"]: item["expected_answer"]
    for item in additional_questions_4
}

keyword_map_4 = {
    item["question"]: item["keywords"]
    for item in additional_questions_4
}

type_map_4 = {
    item["question"]: item["question_type"]
    for item in additional_questions_4
}

new_seed_df_4["expected_answer"] = new_seed_df_4["question"].map(
    expected_map_4
)

new_seed_df_4["keywords"] = new_seed_df_4["question"].map(
    keyword_map_4
)

new_seed_df_4["question_type"] = new_seed_df_4["question"].map(
    type_map_4
)

new_seed_df_4["quality_score"] = new_seed_df_4["quality_label"].map(
    quality_score_map
)

new_seed_df_4 = new_seed_df_4[
    [
        "question",
        "expected_answer",
        "candidate_answer",
        "keywords",
        "question_type",
        "quality_label",
        "quality_score"
    ]
]

# Merge
seed_df = pd.concat(
    [seed_df, new_seed_df_4],
    ignore_index=True
)

# Re-create seed IDs
seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (116, 11)

Quality distribution:
quality_label
Excellent    29
Good         29
Average      29
Poor         29
Name: count, dtype: int64

Number of questions: 29


In [111]:
additional_questions_5 = [
    {
        "question": "What is classification in machine learning?",
        "expected_answer": "Classification is a supervised learning task that predicts a categorical class or label for an input based on learned patterns from labeled training data.",
        "keywords": ["classification", "supervised", "categorical", "label", "training data"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is regression in machine learning?",
        "expected_answer": "Regression is a supervised learning task that predicts a continuous numerical value based on relationships learned from training data.",
        "keywords": ["regression", "supervised", "continuous", "numerical", "prediction"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is accuracy in machine learning?",
        "expected_answer": "Accuracy is an evaluation metric that measures the proportion of correct predictions among all predictions made by a classification model.",
        "keywords": ["accuracy", "evaluation", "correct", "predictions", "classification"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is a foreign key in SQL?",
        "expected_answer": "A foreign key is a column or set of columns in one table that references the primary key of another table to establish a relationship between the tables.",
        "keywords": ["foreign key", "primary key", "table", "relationship", "references"],
        "question_type": "SQL"
    },
    {
        "question": "What is abstraction in OOP?",
        "expected_answer": "Abstraction is an OOP concept that hides unnecessary implementation details and exposes only the essential features or behavior of an object.",
        "keywords": ["abstraction", "OOP", "implementation", "details", "essential"],
        "question_type": "OOP"
    }
]

# Check for duplicates before adding
existing_questions = set(seed_df["question"])

new_questions_5 = [
    item for item in additional_questions_5
    if item["question"] not in existing_questions
]

print("New unique questions:", len(new_questions_5))

if len(new_questions_5) < len(additional_questions_5):
    print("\nDuplicate questions found:")
    print([
        item["question"]
        for item in additional_questions_5
        if item["question"] in existing_questions
    ])

New unique questions: 5


In [112]:
new_seed_answers_5 = [
    # Classification
    {
        "question": "What is classification in machine learning?",
        "candidate_answer": "Classification is a supervised learning task that predicts a categorical class or label for an input based on learned patterns from labeled training data.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is classification in machine learning?",
        "candidate_answer": "Classification is a supervised learning technique used to predict categories or labels from labeled training data.",
        "quality_label": "Good"
    },
    {
        "question": "What is classification in machine learning?",
        "candidate_answer": "Classification is used to predict classes or categories.",
        "quality_label": "Average"
    },
    {
        "question": "What is classification in machine learning?",
        "candidate_answer": "Classification predicts a category.",
        "quality_label": "Poor"
    },

    # Regression
    {
        "question": "What is regression in machine learning?",
        "candidate_answer": "Regression is a supervised learning task that predicts a continuous numerical value based on relationships learned from training data.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is regression in machine learning?",
        "candidate_answer": "Regression is a supervised learning technique used to predict continuous numerical values from training data.",
        "quality_label": "Good"
    },
    {
        "question": "What is regression in machine learning?",
        "candidate_answer": "Regression is used to predict numerical values.",
        "quality_label": "Average"
    },
    {
        "question": "What is regression in machine learning?",
        "candidate_answer": "Regression predicts numbers.",
        "quality_label": "Poor"
    },

    # Accuracy
    {
        "question": "What is accuracy in machine learning?",
        "candidate_answer": "Accuracy is an evaluation metric that measures the proportion of correct predictions among all predictions made by a classification model.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is accuracy in machine learning?",
        "candidate_answer": "Accuracy measures the proportion of predictions that a classification model gets correct.",
        "quality_label": "Good"
    },
    {
        "question": "What is accuracy in machine learning?",
        "candidate_answer": "Accuracy tells us how many predictions are correct.",
        "quality_label": "Average"
    },
    {
        "question": "What is accuracy in machine learning?",
        "candidate_answer": "Accuracy measures correctness.",
        "quality_label": "Poor"
    },

    # Foreign Key
    {
        "question": "What is a foreign key in SQL?",
        "candidate_answer": "A foreign key is a column or set of columns in one table that references the primary key of another table to establish a relationship between the tables.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a foreign key in SQL?",
        "candidate_answer": "A foreign key is used to connect tables by referencing a primary key in another table.",
        "quality_label": "Good"
    },
    {
        "question": "What is a foreign key in SQL?",
        "candidate_answer": "A foreign key creates a relationship between tables.",
        "quality_label": "Average"
    },
    {
        "question": "What is a foreign key in SQL?",
        "candidate_answer": "A foreign key connects tables.",
        "quality_label": "Poor"
    },

    # Abstraction
    {
        "question": "What is abstraction in OOP?",
        "candidate_answer": "Abstraction is an OOP concept that hides unnecessary implementation details and exposes only the essential features or behavior of an object.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is abstraction in OOP?",
        "candidate_answer": "Abstraction hides implementation details and exposes only the essential features of an object.",
        "quality_label": "Good"
    },
    {
        "question": "What is abstraction in OOP?",
        "candidate_answer": "Abstraction means hiding unnecessary details and showing important features.",
        "quality_label": "Average"
    },
    {
        "question": "What is abstraction in OOP?",
        "candidate_answer": "Abstraction hides details.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(new_seed_answers_5))

New seed answers: 20


In [113]:
new_seed_df_5 = pd.DataFrame(new_seed_answers_5)

expected_map_5 = {
    item["question"]: item["expected_answer"]
    for item in additional_questions_5
}

keyword_map_5 = {
    item["question"]: item["keywords"]
    for item in additional_questions_5
}

type_map_5 = {
    item["question"]: item["question_type"]
    for item in additional_questions_5
}

new_seed_df_5["expected_answer"] = new_seed_df_5["question"].map(
    expected_map_5
)

new_seed_df_5["keywords"] = new_seed_df_5["question"].map(
    keyword_map_5
)

new_seed_df_5["question_type"] = new_seed_df_5["question"].map(
    type_map_5
)

new_seed_df_5["quality_score"] = new_seed_df_5["quality_label"].map(
    quality_score_map
)

new_seed_df_5 = new_seed_df_5[
    [
        "question",
        "expected_answer",
        "candidate_answer",
        "keywords",
        "question_type",
        "quality_label",
        "quality_score"
    ]
]

seed_df = pd.concat(
    [seed_df, new_seed_df_5],
    ignore_index=True
)

seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (136, 11)

Quality distribution:
quality_label
Excellent    34
Good         34
Average      34
Poor         34
Name: count, dtype: int64

Number of questions: 34


In [114]:
additional_questions_6 = [
    {
        "question": "What is the difference between Series and DataFrame in Pandas?",
        "expected_answer": "A Series is a one-dimensional labeled data structure in Pandas, while a DataFrame is a two-dimensional structure consisting of rows and columns.",
        "keywords": ["Series", "DataFrame", "one-dimensional", "two-dimensional", "rows", "columns"],
        "question_type": "Data Science"
    },
    {
        "question": "What is standard deviation?",
        "expected_answer": "Standard deviation is a statistical measure that describes how much the values in a dataset vary or deviate from the mean.",
        "keywords": ["standard deviation", "statistical", "variation", "deviation", "mean"],
        "question_type": "Statistics"
    },
    {
        "question": "What is a confusion matrix?",
        "expected_answer": "A confusion matrix is a table used to evaluate a classification model by showing true positives, true negatives, false positives, and false negatives.",
        "keywords": ["confusion matrix", "classification", "true positive", "true negative", "false positive", "false negative"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is a SQL query?",
        "expected_answer": "A SQL query is a command written using Structured Query Language to retrieve, insert, update, or delete data in a relational database.",
        "keywords": ["SQL", "query", "retrieve", "insert", "update", "delete"],
        "question_type": "SQL"
    },
    {
        "question": "What is time complexity?",
        "expected_answer": "Time complexity describes how the running time of an algorithm grows as the size of its input increases.",
        "keywords": ["time complexity", "algorithm", "running time", "input", "growth"],
        "question_type": "Algorithms"
    }
]

existing_questions = set(seed_df["question"])

new_questions_6 = [
    item for item in additional_questions_6
    if item["question"] not in existing_questions
]

print("New unique questions:", len(new_questions_6))

if len(new_questions_6) < len(additional_questions_6):
    print("\nDuplicate questions found:")
    print([
        item["question"]
        for item in additional_questions_6
        if item["question"] in existing_questions
    ])

New unique questions: 5


In [115]:
new_seed_answers_6 = [
    # Series vs DataFrame
    {
        "question": "What is the difference between Series and DataFrame in Pandas?",
        "candidate_answer": "A Series is a one-dimensional labeled data structure in Pandas, while a DataFrame is a two-dimensional structure consisting of rows and columns.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is the difference between Series and DataFrame in Pandas?",
        "candidate_answer": "A Series is one-dimensional, while a DataFrame is two-dimensional with rows and columns.",
        "quality_label": "Good"
    },
    {
        "question": "What is the difference between Series and DataFrame in Pandas?",
        "candidate_answer": "A Series stores data in one dimension and a DataFrame stores data in rows and columns.",
        "quality_label": "Average"
    },
    {
        "question": "What is the difference between Series and DataFrame in Pandas?",
        "candidate_answer": "They are different Pandas data structures.",
        "quality_label": "Poor"
    },

    # Standard Deviation
    {
        "question": "What is standard deviation?",
        "candidate_answer": "Standard deviation is a statistical measure that describes how much the values in a dataset vary or deviate from the mean.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is standard deviation?",
        "candidate_answer": "Standard deviation measures how much values in a dataset vary from the mean.",
        "quality_label": "Good"
    },
    {
        "question": "What is standard deviation?",
        "candidate_answer": "It measures the spread or variation of data.",
        "quality_label": "Average"
    },
    {
        "question": "What is standard deviation?",
        "candidate_answer": "It is a statistical measure.",
        "quality_label": "Poor"
    },

    # Confusion Matrix
    {
        "question": "What is a confusion matrix?",
        "candidate_answer": "A confusion matrix is a table used to evaluate a classification model by showing true positives, true negatives, false positives, and false negatives.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a confusion matrix?",
        "candidate_answer": "A confusion matrix evaluates classification predictions using true positives, true negatives, false positives, and false negatives.",
        "quality_label": "Good"
    },
    {
        "question": "What is a confusion matrix?",
        "candidate_answer": "A confusion matrix is used to evaluate a classification model.",
        "quality_label": "Average"
    },
    {
        "question": "What is a confusion matrix?",
        "candidate_answer": "It is a machine learning table.",
        "quality_label": "Poor"
    },

    # SQL Query
    {
        "question": "What is a SQL query?",
        "candidate_answer": "A SQL query is a command written using Structured Query Language to retrieve, insert, update, or delete data in a relational database.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a SQL query?",
        "candidate_answer": "A SQL query is used to retrieve or modify data in a relational database.",
        "quality_label": "Good"
    },
    {
        "question": "What is a SQL query?",
        "candidate_answer": "A query is used to work with data in a database.",
        "quality_label": "Average"
    },
    {
        "question": "What is a SQL query?",
        "candidate_answer": "A SQL query is a database command.",
        "quality_label": "Poor"
    },

    # Time Complexity
    {
        "question": "What is time complexity?",
        "candidate_answer": "Time complexity describes how the running time of an algorithm grows as the size of its input increases.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is time complexity?",
        "candidate_answer": "Time complexity describes how an algorithm's running time changes as the input size increases.",
        "quality_label": "Good"
    },
    {
        "question": "What is time complexity?",
        "candidate_answer": "Time complexity measures the running time of an algorithm based on its input size.",
        "quality_label": "Average"
    },
    {
        "question": "What is time complexity?",
        "candidate_answer": "It measures algorithm speed.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(new_seed_answers_6))

New seed answers: 20


In [116]:
new_seed_df_6 = pd.DataFrame(new_seed_answers_6)

expected_map_6 = {
    item["question"]: item["expected_answer"]
    for item in additional_questions_6
}

keyword_map_6 = {
    item["question"]: item["keywords"]
    for item in additional_questions_6
}

type_map_6 = {
    item["question"]: item["question_type"]
    for item in additional_questions_6
}

new_seed_df_6["expected_answer"] = new_seed_df_6["question"].map(
    expected_map_6
)

new_seed_df_6["keywords"] = new_seed_df_6["question"].map(
    keyword_map_6
)

new_seed_df_6["question_type"] = new_seed_df_6["question"].map(
    type_map_6
)

new_seed_df_6["quality_score"] = new_seed_df_6["quality_label"].map(
    quality_score_map
)

new_seed_df_6 = new_seed_df_6[
    [
        "question",
        "expected_answer",
        "candidate_answer",
        "keywords",
        "question_type",
        "quality_label",
        "quality_score"
    ]
]

seed_df = pd.concat(
    [seed_df, new_seed_df_6],
    ignore_index=True
)

seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (156, 11)

Quality distribution:
quality_label
Excellent    39
Good         39
Average      39
Poor         39
Name: count, dtype: int64

Number of questions: 39


In [117]:
additional_questions_7 = [
    {
        "question": "What is precision in machine learning?",
        "expected_answer": "Precision is a classification metric that measures the proportion of predicted positive cases that are actually positive.",
        "keywords": ["precision", "classification", "predicted positive", "actual positive", "metric"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is recall in machine learning?",
        "expected_answer": "Recall is a classification metric that measures the proportion of actual positive cases that are correctly identified by the model.",
        "keywords": ["recall", "classification", "actual positive", "correctly identified", "metric"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is a histogram?",
        "expected_answer": "A histogram is a graphical representation that shows the distribution of numerical data by grouping values into intervals called bins.",
        "keywords": ["histogram", "distribution", "numerical", "bins", "data"],
        "question_type": "Statistics"
    },
    {
        "question": "What is an INNER JOIN in SQL?",
        "expected_answer": "An INNER JOIN returns only the rows that have matching values in both tables based on the specified join condition.",
        "keywords": ["INNER JOIN", "matching", "rows", "tables", "condition"],
        "question_type": "SQL"
    },
    {
        "question": "What is inheritance in OOP?",
        "expected_answer": "Inheritance is an OOP concept that allows a child class to acquire properties and methods from a parent class, promoting code reuse.",
        "keywords": ["inheritance", "OOP", "child class", "parent class", "code reuse"],
        "question_type": "OOP"
    }
]

# Check for duplicates
existing_questions = set(seed_df["question"])

new_questions_7 = [
    item for item in additional_questions_7
    if item["question"] not in existing_questions
]

print("New unique questions:", len(new_questions_7))

if len(new_questions_7) < len(additional_questions_7):
    print("\nDuplicate questions found:")
    print([
        item["question"]
        for item in additional_questions_7
        if item["question"] in existing_questions
    ])

New unique questions: 4

Duplicate questions found:
['What is inheritance in OOP?']


In [118]:
new_seed_answers_7 = [
    # Precision
    {
        "question": "What is precision in machine learning?",
        "candidate_answer": "Precision is a classification metric that measures the proportion of predicted positive cases that are actually positive.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is precision in machine learning?",
        "candidate_answer": "Precision measures how many of the cases predicted as positive are actually positive.",
        "quality_label": "Good"
    },
    {
        "question": "What is precision in machine learning?",
        "candidate_answer": "Precision measures the correctness of positive predictions.",
        "quality_label": "Average"
    },
    {
        "question": "What is precision in machine learning?",
        "candidate_answer": "Precision measures positive predictions.",
        "quality_label": "Poor"
    },

    # Recall
    {
        "question": "What is recall in machine learning?",
        "candidate_answer": "Recall is a classification metric that measures the proportion of actual positive cases that are correctly identified by the model.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is recall in machine learning?",
        "candidate_answer": "Recall measures how many of the actual positive cases are correctly identified by the model.",
        "quality_label": "Good"
    },
    {
        "question": "What is recall in machine learning?",
        "candidate_answer": "Recall measures how well a model identifies positive cases.",
        "quality_label": "Average"
    },
    {
        "question": "What is recall in machine learning?",
        "candidate_answer": "Recall identifies positive cases.",
        "quality_label": "Poor"
    },

    # Histogram
    {
        "question": "What is a histogram?",
        "candidate_answer": "A histogram is a graphical representation that shows the distribution of numerical data by grouping values into intervals called bins.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a histogram?",
        "candidate_answer": "A histogram shows the distribution of numerical data by grouping values into different intervals or bins.",
        "quality_label": "Good"
    },
    {
        "question": "What is a histogram?",
        "candidate_answer": "A histogram is a graph used to show the distribution of data.",
        "quality_label": "Average"
    },
    {
        "question": "What is a histogram?",
        "candidate_answer": "A histogram is a type of graph.",
        "quality_label": "Poor"
    },

    # INNER JOIN
    {
        "question": "What is an INNER JOIN in SQL?",
        "candidate_answer": "An INNER JOIN returns only the rows that have matching values in both tables based on the specified join condition.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is an INNER JOIN in SQL?",
        "candidate_answer": "An INNER JOIN returns matching rows from two tables based on a specified condition.",
        "quality_label": "Good"
    },
    {
        "question": "What is an INNER JOIN in SQL?",
        "candidate_answer": "An INNER JOIN combines matching data from two tables.",
        "quality_label": "Average"
    },
    {
        "question": "What is an INNER JOIN in SQL?",
        "candidate_answer": "It joins two tables.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(new_seed_answers_7))

New seed answers: 16


In [119]:
new_seed_df_7 = pd.DataFrame(new_seed_answers_7)

expected_map_7 = {
    item["question"]: item["expected_answer"]
    for item in additional_questions_7
}

keyword_map_7 = {
    item["question"]: item["keywords"]
    for item in additional_questions_7
}

type_map_7 = {
    item["question"]: item["question_type"]
    for item in additional_questions_7
}

new_seed_df_7["expected_answer"] = new_seed_df_7["question"].map(
    expected_map_7
)

new_seed_df_7["keywords"] = new_seed_df_7["question"].map(
    keyword_map_7
)

new_seed_df_7["question_type"] = new_seed_df_7["question"].map(
    type_map_7
)

new_seed_df_7["quality_score"] = new_seed_df_7["quality_label"].map(
    quality_score_map
)

new_seed_df_7 = new_seed_df_7[
    [
        "question",
        "expected_answer",
        "candidate_answer",
        "keywords",
        "question_type",
        "quality_label",
        "quality_score"
    ]
]

seed_df = pd.concat(
    [seed_df, new_seed_df_7],
    ignore_index=True
)

seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (172, 11)

Quality distribution:
quality_label
Excellent    43
Good         43
Average      43
Poor         43
Name: count, dtype: int64

Number of questions: 43


In [120]:
additional_questions_8 = [
    {
        "question": "What is a machine learning model?",
        "expected_answer": "A machine learning model is a mathematical or computational representation learned from data that can make predictions or decisions on new inputs.",
        "keywords": ["machine learning", "model", "data", "predictions", "inputs"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is cross-validation?",
        "expected_answer": "Cross-validation is a model evaluation technique that divides data into multiple subsets and repeatedly trains and evaluates the model on different training and validation portions.",
        "keywords": ["cross-validation", "evaluation", "training", "validation", "subsets"],
        "question_type": "Machine Learning"
    },
    {
        "question": "What is feature engineering?",
        "expected_answer": "Feature engineering is the process of creating, transforming, or selecting input features from raw data to improve the performance of a machine learning model.",
        "keywords": ["feature engineering", "features", "raw data", "transforming", "machine learning"],
        "question_type": "Data Science"
    },
    {
        "question": "What is an outlier?",
        "expected_answer": "An outlier is a data point that is unusually different from the other observations in a dataset and may affect statistical analysis or machine learning models.",
        "keywords": ["outlier", "data point", "observations", "dataset", "statistical"],
        "question_type": "Statistics"
    },
    {
        "question": "What is a SQL GROUP BY clause?",
        "expected_answer": "The SQL GROUP BY clause groups rows that have the same values in specified columns and is commonly used with aggregate functions such as COUNT, SUM, and AVG.",
        "keywords": ["GROUP BY", "SQL", "groups", "aggregate", "COUNT", "SUM", "AVG"],
        "question_type": "SQL"
    },
    {
        "question": "What is method overriding in OOP?",
        "expected_answer": "Method overriding occurs when a child class provides its own implementation of a method that is already defined in its parent class.",
        "keywords": ["method overriding", "child class", "parent class", "implementation", "method"],
        "question_type": "OOP"
    },
    {
        "question": "What is the difference between supervised and unsupervised learning?",
        "expected_answer": "Supervised learning uses labeled data to learn a mapping between inputs and outputs, while unsupervised learning works with unlabeled data to discover patterns or structures.",
        "keywords": ["supervised", "unsupervised", "labeled", "unlabeled", "patterns"],
        "question_type": "Machine Learning"
    }
]

# Check for duplicates
existing_questions = set(seed_df["question"])

new_questions_8 = [
    item for item in additional_questions_8
    if item["question"] not in existing_questions
]

print("New unique questions:", len(new_questions_8))

if len(new_questions_8) < len(additional_questions_8):
    print("\nDuplicate questions found:")
    print([
        item["question"]
        for item in additional_questions_8
        if item["question"] in existing_questions
    ])

New unique questions: 6

Duplicate questions found:
['What is feature engineering?']


In [121]:
new_seed_answers_8 = [
    # Machine Learning Model
    {
        "question": "What is a machine learning model?",
        "candidate_answer": "A machine learning model is a mathematical or computational representation learned from data that can make predictions or decisions on new inputs.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a machine learning model?",
        "candidate_answer": "A machine learning model learns patterns from data and uses them to make predictions on new inputs.",
        "quality_label": "Good"
    },
    {
        "question": "What is a machine learning model?",
        "candidate_answer": "A machine learning model learns from data to make predictions.",
        "quality_label": "Average"
    },
    {
        "question": "What is a machine learning model?",
        "candidate_answer": "It is a model that learns from data.",
        "quality_label": "Poor"
    },

    # Cross Validation
    {
        "question": "What is cross-validation?",
        "candidate_answer": "Cross-validation is a model evaluation technique that divides data into multiple subsets and repeatedly trains and evaluates the model on different training and validation portions.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is cross-validation?",
        "candidate_answer": "Cross-validation evaluates a model by dividing data into several subsets and using different subsets for training and validation.",
        "quality_label": "Good"
    },
    {
        "question": "What is cross-validation?",
        "candidate_answer": "Cross-validation is used to evaluate machine learning models using different portions of data.",
        "quality_label": "Average"
    },
    {
        "question": "What is cross-validation?",
        "candidate_answer": "It is a model evaluation technique.",
        "quality_label": "Poor"
    },

    # Outlier
    {
        "question": "What is an outlier?",
        "candidate_answer": "An outlier is a data point that is unusually different from the other observations in a dataset and may affect statistical analysis or machine learning models.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is an outlier?",
        "candidate_answer": "An outlier is an observation that is significantly different from most other values in a dataset.",
        "quality_label": "Good"
    },
    {
        "question": "What is an outlier?",
        "candidate_answer": "An outlier is an unusual value in a dataset.",
        "quality_label": "Average"
    },
    {
        "question": "What is an outlier?",
        "candidate_answer": "It is an unusual data value.",
        "quality_label": "Poor"
    },

    # GROUP BY
    {
        "question": "What is a SQL GROUP BY clause?",
        "candidate_answer": "The SQL GROUP BY clause groups rows that have the same values in specified columns and is commonly used with aggregate functions such as COUNT, SUM, and AVG.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is a SQL GROUP BY clause?",
        "candidate_answer": "GROUP BY groups rows with the same values and is often used with aggregate functions such as COUNT and SUM.",
        "quality_label": "Good"
    },
    {
        "question": "What is a SQL GROUP BY clause?",
        "candidate_answer": "GROUP BY is used to group similar rows in a SQL query.",
        "quality_label": "Average"
    },
    {
        "question": "What is a SQL GROUP BY clause?",
        "candidate_answer": "It groups data in SQL.",
        "quality_label": "Poor"
    },

    # Method Overriding
    {
        "question": "What is method overriding in OOP?",
        "candidate_answer": "Method overriding occurs when a child class provides its own implementation of a method that is already defined in its parent class.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is method overriding in OOP?",
        "candidate_answer": "Method overriding allows a child class to provide a different implementation of a method inherited from its parent class.",
        "quality_label": "Good"
    },
    {
        "question": "What is method overriding in OOP?",
        "candidate_answer": "Method overriding means redefining a parent class method in a child class.",
        "quality_label": "Average"
    },
    {
        "question": "What is method overriding in OOP?",
        "candidate_answer": "It changes a method in a child class.",
        "quality_label": "Poor"
    },

    # Supervised vs Unsupervised
    {
        "question": "What is the difference between supervised and unsupervised learning?",
        "candidate_answer": "Supervised learning uses labeled data to learn a mapping between inputs and outputs, while unsupervised learning works with unlabeled data to discover patterns or structures.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is the difference between supervised and unsupervised learning?",
        "candidate_answer": "Supervised learning uses labeled data, while unsupervised learning uses unlabeled data to find patterns.",
        "quality_label": "Good"
    },
    {
        "question": "What is the difference between supervised and unsupervised learning?",
        "candidate_answer": "Supervised learning learns from labeled data, while unsupervised learning finds patterns in data without labels.",
        "quality_label": "Average"
    },
    {
        "question": "What is the difference between supervised and unsupervised learning?",
        "candidate_answer": "Supervised has labels and unsupervised does not.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(new_seed_answers_8))

New seed answers: 24


In [122]:
new_seed_df_8 = pd.DataFrame(new_seed_answers_8)

expected_map_8 = {
    item["question"]: item["expected_answer"]
    for item in additional_questions_8
}

keyword_map_8 = {
    item["question"]: item["keywords"]
    for item in additional_questions_8
}

type_map_8 = {
    item["question"]: item["question_type"]
    for item in additional_questions_8
}

new_seed_df_8["expected_answer"] = new_seed_df_8["question"].map(
    expected_map_8
)

new_seed_df_8["keywords"] = new_seed_df_8["question"].map(
    keyword_map_8
)

new_seed_df_8["question_type"] = new_seed_df_8["question"].map(
    type_map_8
)

new_seed_df_8["quality_score"] = new_seed_df_8["quality_label"].map(
    quality_score_map
)

new_seed_df_8 = new_seed_df_8[
    [
        "question",
        "expected_answer",
        "candidate_answer",
        "keywords",
        "question_type",
        "quality_label",
        "quality_score"
    ]
]

seed_df = pd.concat(
    [seed_df, new_seed_df_8],
    ignore_index=True
)

seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (196, 11)

Quality distribution:
quality_label
Excellent    49
Good         49
Average      49
Poor         49
Name: count, dtype: int64

Number of questions: 49


In [123]:
additional_question_9 = {
    "question": "What is cosine similarity?",
    "expected_answer": "Cosine similarity measures the similarity between two vectors by calculating the cosine of the angle between them. It is commonly used to compare text embeddings.",
    "keywords": ["cosine similarity", "vectors", "similarity", "angle", "embeddings"],
    "question_type": "NLP"
}

existing_questions = set(seed_df["question"])

if additional_question_9["question"] in existing_questions:
    print("Duplicate question found!")
else:
    print("New question:", additional_question_9["question"])

New question: What is cosine similarity?


In [124]:
cosine_seed_answers = [
    {
        "question": "What is cosine similarity?",
        "candidate_answer": "Cosine similarity measures the similarity between two vectors by calculating the cosine of the angle between them. It is commonly used to compare text embeddings.",
        "quality_label": "Excellent"
    },
    {
        "question": "What is cosine similarity?",
        "candidate_answer": "Cosine similarity measures how similar two vectors are based on the angle between them and is commonly used with text embeddings.",
        "quality_label": "Good"
    },
    {
        "question": "What is cosine similarity?",
        "candidate_answer": "Cosine similarity is used to measure similarity between vectors.",
        "quality_label": "Average"
    },
    {
        "question": "What is cosine similarity?",
        "candidate_answer": "It measures similarity.",
        "quality_label": "Poor"
    }
]

print("New seed answers:", len(cosine_seed_answers))

New seed answers: 4


In [125]:
cosine_df = pd.DataFrame(cosine_seed_answers)

cosine_df["expected_answer"] = additional_question_9["expected_answer"]
cosine_df["keywords"] = [additional_question_9["keywords"]] * len(cosine_df)
cosine_df["question_type"] = additional_question_9["question_type"]

cosine_df["quality_score"] = cosine_df["quality_label"].map(
    quality_score_map
)

cosine_df = cosine_df[
    [
        "question",
        "expected_answer",
        "candidate_answer",
        "keywords",
        "question_type",
        "quality_label",
        "quality_score"
    ]
]

# Add to the seed dataset
seed_df = pd.concat(
    [seed_df, cosine_df],
    ignore_index=True
)

# Create final unique seed IDs
seed_df["seed_id"] = range(1, len(seed_df) + 1)

print("Seed dataset shape:", seed_df.shape)

print("\nQuality distribution:")
print(seed_df["quality_label"].value_counts())

print("\nNumber of questions:", seed_df["question"].nunique())

Seed dataset shape: (200, 11)

Quality distribution:
quality_label
Excellent    50
Good         50
Average      50
Poor         50
Name: count, dtype: int64

Number of questions: 50


In [126]:
seed_columns = [
    "seed_id",
    "question",
    "expected_answer",
    "candidate_answer",
    "keywords",
    "question_type",
    "quality_label",
    "quality_score"
]

seed_df = seed_df[seed_columns].copy()

print("Seed dataset shape:", seed_df.shape)

print("\nColumns:")
print(seed_df.columns.tolist())

print("\nMissing values:")
print(seed_df.isnull().sum())

Seed dataset shape: (200, 8)

Columns:
['seed_id', 'question', 'expected_answer', 'candidate_answer', 'keywords', 'question_type', 'quality_label', 'quality_score']

Missing values:
seed_id             0
question            0
expected_answer     0
candidate_answer    0
keywords            0
question_type       0
quality_label       0
quality_score       0
dtype: int64


In [127]:
seed_df.to_csv("interview_dataset_seed_200.csv", index=False)

print("Seed dataset saved successfully.")

import os
print("File exists:", os.path.exists("interview_dataset_seed_200.csv"))

Seed dataset saved successfully.
File exists: True


In [128]:
augmentation_test = seed_df.head(10).copy()

augmentation_test["variation_id"] = 0

augmentation_test[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label",
        "variation_id"
    ]
]

,seed_id,question,candidate_answer,quality_label,variation_id
0,1,What is Python?,Python is a high-level programming language kn...,Excellent,0
1,2,What is Python?,Python is a high-level programming language wi...,Good,0
2,3,What is Python?,Python is a programming language used to devel...,Average,0
3,4,What is Python?,Python is a language used for programming.,Poor,0
4,5,What is a list in Python?,A list is a mutable and ordered collection in ...,Excellent,0
5,6,What is a list in Python?,A Python list is an ordered collection that ca...,Good,0
6,7,What is a list in Python?,A list stores multiple values in Python.,Average,0
7,8,What is a list in Python?,A list is something used in Python.,Poor,0
8,9,What is machine learning?,Machine learning is a branch of artificial int...,Excellent,0
9,10,What is machine learning?,Machine learning is a part of artificial intel...,Good,0


In [129]:
variation_1 = augmentation_test.copy()

variation_1["variation_id"] = 1

variation_1["candidate_answer"] = [
    "Python is a high-level programming language that is widely used because of its simple syntax and versatility.",
    
    "In Python, a list is an ordered and mutable collection that can contain multiple values.",
    
    "Machine learning is a branch of artificial intelligence where computers learn patterns from data to make predictions.",
    
    "Python is a programming language.",
    
    "A list in Python is an ordered collection that can be modified after it is created.",
    
    "A Python list stores multiple values in an ordered and mutable structure.",
    
    "A list can store several values in Python.",
    
    "A list is used to store data in Python.",
    
    "Machine learning allows computers to learn patterns from data and use them to make predictions.",
    
    "Machine learning is a part of artificial intelligence that enables systems to learn from data."
]

variation_1[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label",
        "variation_id"
    ]
]

,seed_id,question,candidate_answer,quality_label,variation_id
0,1,What is Python?,Python is a high-level programming language th...,Excellent,1
1,2,What is Python?,"In Python, a list is an ordered and mutable co...",Good,1
2,3,What is Python?,Machine learning is a branch of artificial int...,Average,1
3,4,What is Python?,Python is a programming language.,Poor,1
4,5,What is a list in Python?,A list in Python is an ordered collection that...,Excellent,1
5,6,What is a list in Python?,A Python list stores multiple values in an ord...,Good,1
6,7,What is a list in Python?,A list can store several values in Python.,Average,1
7,8,What is a list in Python?,A list is used to store data in Python.,Poor,1
8,9,What is machine learning?,Machine learning allows computers to learn pat...,Excellent,1
9,10,What is machine learning?,Machine learning is a part of artificial intel...,Good,1


In [130]:
variation_1 = augmentation_test.copy()

variation_1["variation_id"] = 1

variation_1["candidate_answer"] = [
    # Seed 1 - Python - Excellent
    "Python is a high-level programming language that is widely used because of its simple syntax and versatility.",

    # Seed 2 - Python - Good
    "In Python, programming is made easier through its simple syntax and wide range of applications.",

    # Seed 3 - Python - Average
    "Python is a programming language commonly used to develop different types of applications.",

    # Seed 4 - Python - Poor
    "Python is a programming language.",

    # Seed 5 - List - Excellent
    "A list in Python is an ordered and mutable collection that can contain multiple values.",

    # Seed 6 - List - Good
    "A Python list stores multiple values in an ordered and mutable collection.",

    # Seed 7 - List - Average
    "A list can store several values in Python.",

    # Seed 8 - List - Poor
    "A list is used to store data in Python.",

    # Seed 9 - Machine Learning - Excellent
    "Machine learning is a branch of artificial intelligence where computers learn patterns from data to make predictions.",

    # Seed 10 - Machine Learning - Good
    "Machine learning is a part of artificial intelligence that enables systems to learn from data."
]

variation_1[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label",
        "variation_id"
    ]
]

,seed_id,question,candidate_answer,quality_label,variation_id
0,1,What is Python?,Python is a high-level programming language th...,Excellent,1
1,2,What is Python?,"In Python, programming is made easier through ...",Good,1
2,3,What is Python?,Python is a programming language commonly used...,Average,1
3,4,What is Python?,Python is a programming language.,Poor,1
4,5,What is a list in Python?,A list in Python is an ordered and mutable col...,Excellent,1
5,6,What is a list in Python?,A Python list stores multiple values in an ord...,Good,1
6,7,What is a list in Python?,A list can store several values in Python.,Average,1
7,8,What is a list in Python?,A list is used to store data in Python.,Poor,1
8,9,What is machine learning?,Machine learning is a branch of artificial int...,Excellent,1
9,10,What is machine learning?,Machine learning is a part of artificial intel...,Good,1


In [131]:
variation_2 = augmentation_test.copy()

variation_2["variation_id"] = 2

variation_2_answers = {
    1: "Python is a versatile, high-level programming language known for its readable syntax and wide range of applications.",
    2: "Python is a popular programming language because it has simple syntax and can be used for many types of development.",
    3: "Python is a programming language used to create different kinds of software.",
    4: "Python is used for programming.",

    5: "A Python list is an ordered, mutable collection that allows multiple values to be stored and modified.",
    6: "A list is an ordered and changeable collection used to store multiple values in Python.",
    7: "Python lists can contain multiple values in an ordered collection.",
    8: "A list stores multiple values.",

    9: "Machine learning is a field of artificial intelligence in which systems learn patterns from data and use those patterns to make predictions.",
    10: "Machine learning is an area of artificial intelligence where computers learn from data."
}

variation_2["candidate_answer"] = variation_2["seed_id"].map(
    variation_2_answers
)

variation_2[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label",
        "variation_id"
    ]
]

,seed_id,question,candidate_answer,quality_label,variation_id
0,1,What is Python?,"Python is a versatile, high-level programming ...",Excellent,2
1,2,What is Python?,Python is a popular programming language becau...,Good,2
2,3,What is Python?,Python is a programming language used to creat...,Average,2
3,4,What is Python?,Python is used for programming.,Poor,2
4,5,What is a list in Python?,"A Python list is an ordered, mutable collectio...",Excellent,2
5,6,What is a list in Python?,A list is an ordered and changeable collection...,Good,2
6,7,What is a list in Python?,Python lists can contain multiple values in an...,Average,2
7,8,What is a list in Python?,A list stores multiple values.,Poor,2
8,9,What is machine learning?,Machine learning is a field of artificial inte...,Excellent,2
9,10,What is machine learning?,Machine learning is an area of artificial inte...,Good,2


In [132]:
variation_0 = augmentation_test.copy()
variation_0["variation_id"] = 0

test_augmented_df = pd.concat(
    [
        variation_0,
        variation_1,
        variation_2
    ],
    ignore_index=True
)

print("Test augmented shape:", test_augmented_df.shape)

print("\nVariation distribution:")
print(test_augmented_df["variation_id"].value_counts())

print("\nQuality distribution:")
print(test_augmented_df["quality_label"].value_counts())

print("\nDuplicate candidate answers:",
      test_augmented_df["candidate_answer"].duplicated().sum())

Test augmented shape: (30, 9)

Variation distribution:
variation_id
0    10
1    10
2    10
Name: count, dtype: int64

Quality distribution:
quality_label
Excellent    9
Good         9
Average      6
Poor         6
Name: count, dtype: int64

Duplicate candidate answers: 0


In [133]:
variation_3 = augmentation_test.copy()

variation_3["variation_id"] = 3

variation_3_answers = {
    1: "Because of its readable syntax and flexibility, Python is a high-level programming language used in many areas of software development.",
    2: "Python is widely used because its syntax is simple and it supports many different programming applications.",
    3: "Used for developing different types of software, Python is a popular programming language.",
    4: "Python can be used for programming.",

    5: "In Python, a list is an ordered collection that can be modified and can contain multiple values.",
    6: "Multiple values can be stored in a Python list, which is ordered and mutable.",
    7: "Several values can be stored in an ordered Python list.",
    8: "Python lists can store data.",

    9: "By learning patterns from data, machine learning systems can make predictions and is considered a branch of artificial intelligence.",
    10: "Systems can learn from data in machine learning, which is a field of artificial intelligence."
}

variation_3["candidate_answer"] = variation_3["seed_id"].map(
    variation_3_answers
)

variation_3[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label",
        "variation_id"
    ]
]

,seed_id,question,candidate_answer,quality_label,variation_id
0,1,What is Python?,Because of its readable syntax and flexibility...,Excellent,3
1,2,What is Python?,Python is widely used because its syntax is si...,Good,3
2,3,What is Python?,Used for developing different types of softwar...,Average,3
3,4,What is Python?,Python can be used for programming.,Poor,3
4,5,What is a list in Python?,"In Python, a list is an ordered collection tha...",Excellent,3
5,6,What is a list in Python?,Multiple values can be stored in a Python list...,Good,3
6,7,What is a list in Python?,Several values can be stored in an ordered Pyt...,Average,3
7,8,What is a list in Python?,Python lists can store data.,Poor,3
8,9,What is machine learning?,"By learning patterns from data, machine learni...",Excellent,3
9,10,What is machine learning?,Systems can learn from data in machine learnin...,Good,3


In [134]:
variation_3.loc[
    variation_3["seed_id"] == 9,
    "candidate_answer"
] = "Machine learning is a branch of artificial intelligence in which systems learn patterns from data and use them to make predictions."

variation_3[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label",
        "variation_id"
    ]
]

,seed_id,question,candidate_answer,quality_label,variation_id
0,1,What is Python?,Because of its readable syntax and flexibility...,Excellent,3
1,2,What is Python?,Python is widely used because its syntax is si...,Good,3
2,3,What is Python?,Used for developing different types of softwar...,Average,3
3,4,What is Python?,Python can be used for programming.,Poor,3
4,5,What is a list in Python?,"In Python, a list is an ordered collection tha...",Excellent,3
5,6,What is a list in Python?,Multiple values can be stored in a Python list...,Good,3
6,7,What is a list in Python?,Several values can be stored in an ordered Pyt...,Average,3
7,8,What is a list in Python?,Python lists can store data.,Poor,3
8,9,What is machine learning?,Machine learning is a branch of artificial int...,Excellent,3
9,10,What is machine learning?,Systems can learn from data in machine learnin...,Good,3


In [135]:
variation_4 = augmentation_test.copy()

variation_4["variation_id"] = 4

variation_4_answers = {
    1: "In an interview, I would describe Python as a high-level programming language with readable syntax that is used across many areas of software development.",
    2: "Python is popular because it is easy to read and supports many different programming tasks and applications.",
    3: "Python is a widely used programming language for developing software and applications.",
    4: "Python is a programming language.",

    5: "A list in Python is an ordered and mutable collection, so its elements can be changed after the list is created.",
    6: "Python lists are ordered collections that can be modified and can store multiple values.",
    7: "A list is an ordered collection that can hold multiple values in Python.",
    8: "A list stores values in Python.",

    9: "Machine learning is a branch of artificial intelligence where a system learns patterns from data and uses those patterns to make predictions.",
    10: "Machine learning allows computers to learn from data and is a field within artificial intelligence."
}

variation_4["candidate_answer"] = variation_4["seed_id"].map(
    variation_4_answers
)

variation_4[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label",
        "variation_id"
    ]
]

,seed_id,question,candidate_answer,quality_label,variation_id
0,1,What is Python?,"In an interview, I would describe Python as a ...",Excellent,4
1,2,What is Python?,Python is popular because it is easy to read a...,Good,4
2,3,What is Python?,Python is a widely used programming language f...,Average,4
3,4,What is Python?,Python is a programming language.,Poor,4
4,5,What is a list in Python?,A list in Python is an ordered and mutable col...,Excellent,4
5,6,What is a list in Python?,Python lists are ordered collections that can ...,Good,4
6,7,What is a list in Python?,A list is an ordered collection that can hold ...,Average,4
7,8,What is a list in Python?,A list stores values in Python.,Poor,4
8,9,What is machine learning?,Machine learning is a branch of artificial int...,Excellent,4
9,10,What is machine learning?,Machine learning allows computers to learn fro...,Good,4


In [136]:
variation_0 = augmentation_test.copy()
variation_0["variation_id"] = 0

test_augmented_df = pd.concat(
    [
        variation_0,
        variation_1,
        variation_2,
        variation_3,
        variation_4
    ],
    ignore_index=True
)

print("Dataset shape:", test_augmented_df.shape)

print("\nVariation distribution:")
print(test_augmented_df["variation_id"].value_counts().sort_index())

print("\nQuality distribution:")
print(test_augmented_df["quality_label"].value_counts())

print("\nDuplicate candidate answers:",
      test_augmented_df["candidate_answer"].duplicated().sum())

print("\nMissing values:")
print(test_augmented_df.isnull().sum())

Dataset shape: (50, 9)

Variation distribution:
variation_id
0    10
1    10
2    10
3    10
4    10
Name: count, dtype: int64

Quality distribution:
quality_label
Excellent    15
Good         15
Average      10
Poor         10
Name: count, dtype: int64

Duplicate candidate answers: 1

Missing values:
seed_id             0
question            0
expected_answer     0
candidate_answer    0
keywords            0
question_type       0
quality_label       0
quality_score       0
variation_id        0
dtype: int64


In [137]:
duplicates = test_augmented_df[
    test_augmented_df["candidate_answer"].duplicated(keep=False)
].sort_values("candidate_answer")

duplicates[
    [
        "seed_id",
        "question",
        "candidate_answer",
        "quality_label",
        "variation_id"
    ]
]

,seed_id,question,candidate_answer,quality_label,variation_id
13,4,What is Python?,Python is a programming language.,Poor,1
43,4,What is Python?,Python is a programming language.,Poor,4


In [138]:
variation_4.loc[
    variation_4["seed_id"] == 4,
    "candidate_answer"
] = "Python is used to write programs."

In [139]:
test_augmented_df = pd.concat(
    [
        variation_0,
        variation_1,
        variation_2,
        variation_3,
        variation_4
    ],
    ignore_index=True
)

print("Dataset shape:", test_augmented_df.shape)

print("\nVariation distribution:")
print(
    test_augmented_df["variation_id"]
    .value_counts()
    .sort_index()
)

print("\nQuality distribution:")
print(
    test_augmented_df["quality_label"]
    .value_counts()
)

print("\nDuplicate candidate answers:",
      test_augmented_df["candidate_answer"].duplicated().sum())

print("\nMissing values:")
print(test_augmented_df.isnull().sum())

Dataset shape: (50, 9)

Variation distribution:
variation_id
0    10
1    10
2    10
3    10
4    10
Name: count, dtype: int64

Quality distribution:
quality_label
Excellent    15
Good         15
Average      10
Poor         10
Name: count, dtype: int64

Duplicate candidate answers: 0

Missing values:
seed_id             0
question            0
expected_answer     0
candidate_answer    0
keywords            0
question_type       0
quality_label       0
quality_score       0
variation_id        0
dtype: int64


In [142]:
def create_variations(answer):
    """
    Create controlled variations while preserving
    the original technical answer.
    """

    variations = [
        answer,

        f"In general, {answer}",

        f"In simple terms, {answer}",

        f"From a technical perspective, {answer}",

        f"Overall, {answer}"
    ]

    return variations

In [143]:
test_answer = seed_df.iloc[0]["candidate_answer"]

test_variations = create_variations(test_answer)

for i, variation in enumerate(test_variations):
    print(f"Variation {i}:")
    print(variation)
    print()

Variation 0:
Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.

Variation 1:
In general, Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.

Variation 2:
In simple terms, Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.

Variation 3:
From a technical perspective, Python is a high-level programming language known for its simple and readable syntax. It is widely used in web development, automation, data science, artificial intelligence, and machine learning.

Variation 4:
Overall, Python is a high-level programming language known for its simple and readab

In [144]:
def create_variations(answer):
    variations = [
        answer,
        f"In general, {answer}",
        f"In simple terms, {answer}",
        f"From a technical perspective, {answer}",
        f"Overall, {answer}"
    ]

    return variations

In [145]:
augmented_rows = []

for _, row in seed_df.iterrows():

    variations = create_variations(row["candidate_answer"])

    for variation_id, answer in enumerate(variations):

        augmented_rows.append({
            "seed_id": row["seed_id"],
            "variation_id": variation_id,
            "question": row["question"],
            "expected_answer": row["expected_answer"],
            "candidate_answer": answer,
            "keywords": row["keywords"],
            "question_type": row["question_type"],
            "quality_label": row["quality_label"],
            "quality_score": row["quality_score"]
        })

augmented_df = pd.DataFrame(augmented_rows)

print("Augmented dataset shape:", augmented_df.shape)

print("\nVariation distribution:")
print(augmented_df["variation_id"].value_counts().sort_index())

print("\nQuality distribution:")
print(augmented_df["quality_label"].value_counts())

Augmented dataset shape: (1000, 9)

Variation distribution:
variation_id
0    200
1    200
2    200
3    200
4    200
Name: count, dtype: int64

Quality distribution:
quality_label
Excellent    250
Good         250
Average      250
Poor         250
Name: count, dtype: int64


In [146]:
print("Dataset shape:", augmented_df.shape)

print("\nMissing values:")
print(augmented_df.isnull().sum())

print("\nDuplicate candidate answers:",
      augmented_df["candidate_answer"].duplicated().sum())

print("\nUnique questions:",
      augmented_df["question"].nunique())

print("\nUnique seeds:",
      augmented_df["seed_id"].nunique())

print("\nQuality distribution:")
print(augmented_df["quality_label"].value_counts())

print("\nVariation distribution:")
print(
    augmented_df["variation_id"]
    .value_counts()
    .sort_index()
)

Dataset shape: (1000, 9)

Missing values:
seed_id             0
variation_id        0
question            0
expected_answer     0
candidate_answer    0
keywords            0
question_type       0
quality_label       0
quality_score       0
dtype: int64

Duplicate candidate answers: 0

Unique questions: 50

Unique seeds: 200

Quality distribution:
quality_label
Excellent    250
Good         250
Average      250
Poor         250
Name: count, dtype: int64

Variation distribution:
variation_id
0    200
1    200
2    200
3    200
4    200
Name: count, dtype: int64


In [147]:
# Save the final 1000-row dataset
augmented_df.to_csv(
    "interview_dataset_1000.csv",
    index=False
)

print("Final dataset saved successfully.")

import os
print(
    "File exists:",
    os.path.exists("interview_dataset_1000.csv")
)

Final dataset saved successfully.
File exists: True


In [148]:
import pandas as pd

df = pd.read_csv("interview_dataset_1000.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1000, 9)

Columns:
['seed_id', 'variation_id', 'question', 'expected_answer', 'candidate_answer', 'keywords', 'question_type', 'quality_label', 'quality_score']


In [149]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2122.28it/s]


Embedding model loaded successfully.


In [150]:
expected_embeddings = embedding_model.encode(
    df["expected_answer"].tolist(),
    show_progress_bar=True
)

candidate_embeddings = embedding_model.encode(
    df["candidate_answer"].tolist(),
    show_progress_bar=True
)

print("Expected embeddings shape:", expected_embeddings.shape)
print("Candidate embeddings shape:", candidate_embeddings.shape)

Batches: 100%|██████████| 32/32 [00:04<00:00,  7.87it/s]


Expected embeddings shape: (1000, 384)
Candidate embeddings shape: (1000, 384)


In [151]:
from sklearn.metrics.pairwise import cosine_similarity

df["semantic_score"] = [
    cosine_similarity(
        expected_embeddings[i].reshape(1, -1),
        candidate_embeddings[i].reshape(1, -1)
    )[0][0]
    for i in range(len(df))
]

print(df["semantic_score"].describe())

count    1000.000000
mean        0.836946
std         0.149375
min         0.341175
25%         0.780856
50%         0.877610
75%         0.952794
max         1.000000
Name: semantic_score, dtype: float64


In [152]:
def calculate_keyword_features(row):
    answer = row["candidate_answer"].lower()

    keywords = row["keywords"]

    matched = sum(
        1 for keyword in keywords
        if keyword.lower() in answer
    )

    total = len(keywords)

    score = matched / total if total > 0 else 0

    return pd.Series([
        matched,
        total,
        score
    ])


df[
    [
        "matched_keywords",
        "total_keywords",
        "keyword_score"
    ]
] = df.apply(
    calculate_keyword_features,
    axis=1
)

print(df[
    [
        "matched_keywords",
        "total_keywords",
        "keyword_score"
    ]
].head(10))

   matched_keywords  total_keywords  keyword_score
0              61.0            73.0       0.835616
1              61.0            73.0       0.835616
2              61.0            73.0       0.835616
3              61.0            73.0       0.835616
4              61.0            73.0       0.835616
5              56.0            73.0       0.767123
6              60.0            73.0       0.821918
7              60.0            73.0       0.821918
8              60.0            73.0       0.821918
9              60.0            73.0       0.821918


In [153]:
import ast

df["keywords"] = df["keywords"].apply(ast.literal_eval)

print("Example keywords:")
print(df["keywords"].iloc[0])

print("\nNumber of keywords:",
      len(df["keywords"].iloc[0]))

Example keywords:
['python', 'high-level', 'programming language', 'syntax', 'readability']

Number of keywords: 5


In [154]:
df[
    [
        "matched_keywords",
        "total_keywords",
        "keyword_score"
    ]
] = df.apply(
    calculate_keyword_features,
    axis=1
)

print(df[
    [
        "matched_keywords",
        "total_keywords",
        "keyword_score"
    ]
].head(10))

   matched_keywords  total_keywords  keyword_score
0               4.0             5.0            0.8
1               4.0             5.0            0.8
2               4.0             5.0            0.8
3               4.0             5.0            0.8
4               4.0             5.0            0.8
5               4.0             5.0            0.8
6               4.0             5.0            0.8
7               4.0             5.0            0.8
8               4.0             5.0            0.8
9               4.0             5.0            0.8


In [155]:
df["answer_length"] = df["candidate_answer"].apply(
    lambda answer: len(answer.split())
)

df["expected_length"] = df["expected_answer"].apply(
    lambda answer: len(answer.split())
)

df["length_ratio"] = (
    df["answer_length"] /
    df["expected_length"]
)

print(
    df[
        [
            "answer_length",
            "expected_length",
            "length_ratio",
            "quality_label"
        ]
    ].head(10)
)

   answer_length  expected_length  length_ratio quality_label
0             28               13      2.153846     Excellent
1             30               13      2.307692     Excellent
2             31               13      2.384615     Excellent
3             32               13      2.461538     Excellent
4             29               13      2.230769     Excellent
5             21               13      1.615385          Good
6             23               13      1.769231          Good
7             24               13      1.846154          Good
8             25               13      1.923077          Good
9             22               13      1.692308          Good


In [156]:
import re

def calculate_sentence_features(answer):
    sentences = re.split(r'[.!?]+', answer)
    sentences = [s.strip() for s in sentences if s.strip()]

    sentence_count = len(sentences)

    if sentence_count > 0:
        avg_sentence_length = (
            len(answer.split()) / sentence_count
        )
    else:
        avg_sentence_length = 0

    return pd.Series([
        sentence_count,
        avg_sentence_length
    ])


df[
    [
        "sentence_count",
        "avg_sentence_length"
    ]
] = df["candidate_answer"].apply(
    calculate_sentence_features
)

print(
    df[
        [
            "answer_length",
            "sentence_count",
            "avg_sentence_length",
            "quality_label"
        ]
    ].head(10)
)

   answer_length  sentence_count  avg_sentence_length quality_label
0             28             2.0                 14.0     Excellent
1             30             2.0                 15.0     Excellent
2             31             2.0                 15.5     Excellent
3             32             2.0                 16.0     Excellent
4             29             2.0                 14.5     Excellent
5             21             2.0                 10.5          Good
6             23             2.0                 11.5          Good
7             24             2.0                 12.0          Good
8             25             2.0                 12.5          Good
9             22             2.0                 11.0          Good


In [157]:
def calculate_unique_word_ratio(answer):
    words = answer.lower().split()

    if len(words) == 0:
        return 0

    unique_words = set(words)

    return len(unique_words) / len(words)


df["unique_word_ratio"] = df["candidate_answer"].apply(
    calculate_unique_word_ratio
)

print(
    df[
        [
            "answer_length",
            "unique_word_ratio",
            "quality_label"
        ]
    ].head(10)
)

   answer_length  unique_word_ratio quality_label
0             28           0.928571     Excellent
1             30           0.900000     Excellent
2             31           0.870968     Excellent
3             32           0.906250     Excellent
4             29           0.931034     Excellent
5             21           0.904762          Good
6             23           0.869565          Good
7             24           0.833333          Good
8             25           0.880000          Good
9             22           0.909091          Good


In [158]:
feature_columns = [
    "semantic_score",
    "keyword_score",
    "answer_length",
    "length_ratio",
    "matched_keywords",
    "sentence_count",
    "avg_sentence_length",
    "unique_word_ratio"
]

X = df[feature_columns]
y = df["quality_score"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:")
print(feature_columns)

Feature matrix shape: (1000, 8)
Target shape: (1000,)

Features:
['semantic_score', 'keyword_score', 'answer_length', 'length_ratio', 'matched_keywords', 'sentence_count', 'avg_sentence_length', 'unique_word_ratio']


In [159]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["seed_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 800
Testing samples: 200


In [160]:
train_seeds = set(df.iloc[train_idx]["seed_id"])
test_seeds = set(df.iloc[test_idx]["seed_id"])

overlap = train_seeds.intersection(test_seeds)

print("Training seeds:", len(train_seeds))
print("Testing seeds:", len(test_seeds))
print("Overlapping seeds:", len(overlap))
print("Overlap:", overlap)

Training seeds: 160
Testing seeds: 40
Overlapping seeds: 0
Overlap: set()


In [161]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42
    )
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE"
).reset_index(drop=True)

results_df

,Model,MAE,R2
0,Decision Tree,4.500000,0.851240
1,Random Forest,4.624375,0.925150
2,Extra Trees,4.831875,0.913434
3,Gradient Boosting,5.695476,0.912373
4,Linear Regression,7.756846,0.881705


In [162]:
from sklearn.model_selection import GroupKFold, cross_validate

cv = GroupKFold(n_splits=5)

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        groups=df.iloc[train_idx]["seed_id"],
        scoring={
            "MAE": "neg_mean_absolute_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    mean_mae = -scores["test_MAE"].mean()
    mean_r2 = scores["test_R2"].mean()

    cv_results.append({
        "Model": name,
        "Mean MAE": mean_mae,
        "Mean R2": mean_r2
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="Mean MAE"
).reset_index(drop=True)

cv_results_df

,Model,Mean MAE,Mean R2
0,Decision Tree,6.000000,0.803361
1,Random Forest,6.270469,0.871129
2,Extra Trees,6.459844,0.859270
3,Gradient Boosting,6.757562,0.875282
4,Linear Regression,8.397769,0.857352


In [163]:
from sklearn.ensemble import RandomForestRegressor

final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

final_model.fit(X_train, y_train)

print("Final model trained successfully.")

Final model trained successfully.


In [164]:
import joblib

joblib.dump(
    final_model,
    "interview_score_model.pkl"
)

print("Model saved successfully.")

import os
print(
    "File exists:",
    os.path.exists("interview_score_model.pkl")
)

Model saved successfully.
File exists: True


In [165]:
loaded_model = joblib.load(
    "interview_score_model.pkl"
)

test_predictions = loaded_model.predict(
    X_test.iloc[:10]
)

print("Predicted scores:")
print(test_predictions)

print("\nActual scores:")
print(y_test.iloc[:10].values)

Predicted scores:
[85.125 82.125 90.5   90.625 77.375 25.    26.5   28.    28.    25.875]

Actual scores:
[75 75 75 75 75 25 25 25 25 25]


In [166]:
feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": final_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance

,Feature,Importance
0,answer_length,0.740611
1,length_ratio,0.103880
2,semantic_score,0.083335
3,avg_sentence_length,0.028989
4,keyword_score,0.021570
5,unique_word_ratio,0.014962
6,matched_keywords,0.006591
7,sentence_count,0.000061


In [168]:
revised_features = [
    "semantic_score",
    "keyword_score",
    "length_ratio",
    "sentence_count",
    "avg_sentence_length",
    "unique_word_ratio"
]

X_revised = df[revised_features]
y = df["quality_score"]

# Use the exact same train/test groups as before
X_train_revised = X_revised.iloc[train_idx]
X_test_revised = X_revised.iloc[test_idx]

y_train_revised = y.iloc[train_idx]
y_test_revised = y.iloc[test_idx]

revised_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

revised_model.fit(
    X_train_revised,
    y_train_revised
)

revised_predictions = revised_model.predict(
    X_test_revised
)

revised_mae = mean_absolute_error(
    y_test_revised,
    revised_predictions
)

revised_r2 = r2_score(
    y_test_revised,
    revised_predictions
)

print("Revised Random Forest")
print("MAE:", round(revised_mae, 3))
print("R2 :", round(revised_r2, 3))

Revised Random Forest
MAE: 4.876
R2 : 0.911


In [169]:
from sklearn.model_selection import GroupKFold, cross_validate

cv = GroupKFold(n_splits=5)

revised_cv = cross_validate(
    revised_model,
    X_train_revised,
    y_train_revised,
    cv=cv,
    groups=df.iloc[train_idx]["seed_id"],
    scoring={
        "MAE": "neg_mean_absolute_error",
        "R2": "r2"
    },
    n_jobs=-1
)

revised_mean_mae = -revised_cv["test_MAE"].mean()
revised_mean_r2 = revised_cv["test_R2"].mean()

print("Revised Random Forest Cross-Validation")
print("Mean MAE:", round(revised_mean_mae, 3))
print("Mean R2 :", round(revised_mean_r2, 3))

Revised Random Forest Cross-Validation
Mean MAE: 6.467
Mean R2 : 0.863


In [170]:
final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

final_model.fit(
    X,
    y
)

print("Final deployment model trained.")
print("Training samples used:", len(X))

Final deployment model trained.
Training samples used: 1000


In [171]:
import joblib
import os

joblib.dump(
    final_model,
    "interview_score_model.pkl"
)

print("Final model saved successfully.")
print(
    "File exists:",
    os.path.exists("interview_score_model.pkl")
)

Final model saved successfully.
File exists: True


In [172]:
for i, question in enumerate(sorted(df["question"].unique()), start=1):
    print(f"{i}. {question}")

1. What is NumPy?
2. What is Pandas?
3. What is Python?
4. What is SQL?
5. What is a DataFrame in Pandas?
6. What is a JOIN in SQL?
7. What is a Python function?
8. What is a SQL GROUP BY clause?
9. What is a SQL query?
10. What is a confusion matrix?
11. What is a dictionary in Python?
12. What is a foreign key in SQL?
13. What is a histogram?
14. What is a list in Python?
15. What is a machine learning model?
16. What is a primary key in SQL?
17. What is a queue in data structures?
18. What is a stack in data structures?
19. What is a tuple in Python?
20. What is abstraction in OOP?
21. What is accuracy in machine learning?
22. What is an INNER JOIN in SQL?
23. What is an outlier?
24. What is binary search?
25. What is classification in machine learning?
26. What is cosine similarity?
27. What is cross-validation?
28. What is data preprocessing?
29. What is database normalization?
30. What is encapsulation in OOP?
31. What is exploratory data analysis?
32. What is feature engineering

In [173]:
questions_list = sorted(df["question"].unique())

for i, question in enumerate(questions_list[25:50], start=26):
    print(f"{i}. {question}")

26. What is cosine similarity?
27. What is cross-validation?
28. What is data preprocessing?
29. What is database normalization?
30. What is encapsulation in OOP?
31. What is exploratory data analysis?
32. What is feature engineering?
33. What is inheritance in OOP?
34. What is machine learning?
35. What is mean in statistics?
36. What is method overriding in OOP?
37. What is normalization in machine learning?
38. What is object-oriented programming?
39. What is overfitting in machine learning?
40. What is polymorphism in OOP?
41. What is precision in machine learning?
42. What is recall in machine learning?
43. What is regression in machine learning?
44. What is standard deviation?
45. What is supervised learning?
46. What is the difference between Series and DataFrame in Pandas?
47. What is the difference between supervised and unsupervised learning?
48. What is time complexity?
49. What is train-test splitting?
50. What is unsupervised learning?
